In [1]:
# Cell 0 | 设置两模型最终参数确定与模型冻结环境、路径和固定方案

import copy
import gc
import hashlib
import inspect
import json
import os
import random
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import lightgbm
import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display
from lightgbm import LGBMRegressor
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


def find_project_root(start_path="."):
    current = Path(start_path).resolve()

    for _ in range(20):
        if (
            (current / "data").exists()
            and (current / "notebooks").exists()
        ):
            return current

        if current.parent == current:
            break

        current = current.parent

    raise FileNotFoundError(
        "未找到项目根目录。请确认当前Notebook位于：\n"
        r"D:\jupyter\IL_ML_Project\notebooks"
    )


def first_existing_path(candidates, label):
    existing = [
        Path(path)
        for path in candidates
        if Path(path).exists()
    ]

    if not existing:
        raise FileNotFoundError(
            f"未找到{label}。检查过的路径：\n"
            + "\n".join(map(str, candidates))
        )

    return existing[0]


def read_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def read_csv_robust(path, **kwargs):
    last_error = None

    for encoding in [
        "utf-8-sig",
        "utf-8",
        "gb18030",
    ]:
        try:
            dataframe = pd.read_csv(
                path,
                encoding=encoding,
                **kwargs,
            )

            dataframe.columns = (
                dataframe.columns
                .astype(str)
                .str.replace(
                    "\ufeff",
                    "",
                    regex=False,
                )
                .str.strip()
            )

            return dataframe

        except UnicodeDecodeError as error:
            last_error = error

    raise last_error


def read_existing_columns(
    path,
    desired_columns,
    required_columns=None,
):
    header = read_csv_robust(
        path,
        nrows=0,
    )

    available = [
        column
        for column in desired_columns
        if column in header.columns
    ]

    required_columns = (
        required_columns
        or []
    )

    missing = [
        column
        for column in required_columns
        if column not in header.columns
    ]

    if missing:
        raise KeyError(
            f"{Path(path).name}缺少必需字段："
            f"{missing}"
        )

    return read_csv_robust(
        path,
        usecols=available,
    )


def calculate_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def array_sha256(array):
    contiguous = np.ascontiguousarray(
        np.asarray(array)
    )

    return hashlib.sha256(
        contiguous.tobytes()
    ).hexdigest()


def seed_everything(seed=42):
    random.seed(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def to_jsonable(value):
    if isinstance(value, dict):
        return {
            str(key): to_jsonable(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            to_jsonable(item)
            for item in value
        ]

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        np.integer,
    ):
        return int(value)

    if isinstance(
        value,
        np.floating,
    ):
        return float(value)

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    return value


def save_json_atomic(
    data,
    output_path,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_name(
        output_path.name
        + ".tmp"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            to_jsonable(data),
            file,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(
        temporary_path,
        output_path,
    )


def save_csv_atomic(
    dataframe,
    output_path,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_name(
        output_path.name
        + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig",
    )

    os.replace(
        temporary_path,
        output_path,
    )


def save_joblib_atomic(
    obj,
    output_path,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_name(
        output_path.name
        + ".tmp"
    )

    joblib.dump(
        obj,
        temporary_path,
    )

    os.replace(
        temporary_path,
        output_path,
    )


def save_torch_atomic(
    payload,
    output_path,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = output_path.with_name(
        output_path.name
        + ".tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )

    os.replace(
        temporary_path,
        output_path,
    )


def all_finite_memmap(
    path,
    chunk_rows=4096,
):
    array = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )

    for start in range(
        0,
        len(array),
        chunk_rows,
    ):
        if not np.isfinite(
            np.asarray(
                array[
                    start:start + chunk_rows
                ]
            )
        ).all():
            return False

    return True


PROJECT_ROOT = find_project_root(
    Path.cwd()
)

EXPERIMENT_ROOT = (
    PROJECT_ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

PIPELINE_ROOT = (
    EXPERIMENT_ROOT
    / "outputs"
    / "model_training_3models_cv_no_leakage_2splits"
)

ML_READY_PATH = (
    EXPERIMENT_ROOT
    / "ml_ready"
    / "ml_ready_A.csv"
)

GLOBAL_FEATURE_PATH = (
    EXPERIMENT_ROOT
    / "outputs"
    / "representation_compare"
    / "mpnn_features_eta.npy"
)

GLOBAL_FEATURE_ROW_MANIFEST_PATH = (
    EXPERIMENT_ROOT
    / "outputs"
    / "representation_compare"
    / "mpnn_features_eta_row_manifest.csv"
)

MPNN_MODEL_DIR = (
    EXPERIMENT_ROOT
    / "models"
    / "mpnn_feature_extractors"
)

MPNN_MODEL_PATH = (
    MPNN_MODEL_DIR
    / "mpnn_feature_extractor_viscosity.pt"
)

MPNN_TP_SCALER_PATH = (
    MPNN_MODEL_DIR
    / "tp_scaler_viscosity.joblib"
)

MPNN_TARGET_SCALER_PATH = (
    MPNN_MODEL_DIR
    / "target_scaler_viscosity.joblib"
)

MPNN_METADATA_PATH = (
    MPNN_MODEL_DIR
    / "mpnn_feature_extractor_viscosity_metadata.json"
)

FOLDWISE_FEATURE_ROOT = (
    EXPERIMENT_ROOT
    / "outputs"
    / "foldwise_mpnn_cv_features"
    / "IL_split"
)

# 已完成严格嵌套调参的固定签名
UPSTREAM_TUNING_SIGNATURE = (
    "4eafa9d6fbe7c6671f79cb858e3f0a4f"
    "aa39de711cf68495c13dad5656fd5f78"
)

UPSTREAM_TUNING_STUDY_SUFFIX = (
    UPSTREAM_TUNING_SIGNATURE[:12]
)

PARAMETER_HANDOFF_MANIFEST_PATH = (
    PIPELINE_ROOT
    / "hyperparameter_tuning_inner_foldwise_mpnn"
    / "summaries"
    / UPSTREAM_TUNING_STUDY_SUFFIX
    / "outer_evaluation_handoff"
    / "outer_evaluation_parameter_manifest_Data_IL.json"
)

if not PARAMETER_HANDOFF_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "未找到外层评价参数交接manifest：\n"
        f"{PARAMETER_HANDOFF_MANIFEST_PATH}"
    )

FORMAL_FINAL_MANIFEST_PATH = (
    PIPELINE_ROOT
    / "formal_outer_evaluation"
    / "final_summary"
    / "logs"
    / "formal_outer_evaluation_final_manifest_Data_IL.json"
)

FINAL_ROOT = (
    PIPELINE_ROOT
    / "final_two_model_freeze"
)

STUDY_DIR = (
    FINAL_ROOT
    / "studies"
)

TRIAL_DIR = (
    FINAL_ROOT
    / "trial_results"
)

PARAMETER_DIR = (
    FINAL_ROOT
    / "parameters"
)

MODEL_DIR = (
    FINAL_ROOT
    / "models"
)

SCALER_DIR = (
    FINAL_ROOT
    / "scalers"
)

RESULT_DIR = (
    FINAL_ROOT
    / "results"
)

LOG_DIR = (
    FINAL_ROOT
    / "logs"
)

for folder in [
    FINAL_ROOT,
    STUDY_DIR,
    TRIAL_DIR,
    PARAMETER_DIR,
    MODEL_DIR,
    SCALER_DIR,
    RESULT_DIR,
    LOG_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

output_text = str(
    FINAL_ROOT.resolve()
).lower()

if (
    "small_paper_data"
    not in output_text
    or
    "viscosity_10point_holdout"
    not in output_text
):
    raise RuntimeError(
        "最终模型输出目录异常，已停止运行。"
    )

MODELS = [
    "LightGBM",
    "Transformer",
]

FINAL_APPLICATION_MODEL = (
    "LightGBM"
)

SUPPLEMENTARY_CONTROL_MODEL = (
    "Transformer"
)

FINAL_SELECTION_SPLIT = (
    "IL_split"
)

N_FOLDS = 5

N_TRIALS = {
    "LightGBM": 20,
    "Transformer": 10,
}

TRANSFORMER_SEARCH_EPOCHS = 50
TRANSFORMER_FORMAL_EPOCHS = 150
TRANSFORMER_NUM_TOKENS = 8
TRANSFORMER_NOISE_STD = 0.01

TUNING_RANDOM_SEED = 20260725

FINAL_MODEL_SEEDS = {
    "LightGBM": 202607251,
    "Transformer": 202607252,
}

RUN_CODE_VERSION = (
    "17_final_two_model_tuning_and_"
    "freeze_v1_20260725"
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

EXPECTED_ROWS = 16216
EXPECTED_FEATURE_DIM = 130

EXPECTED_HASHES = {
    "ml_ready_A": (
        "bedae296be0e14d654747a7e57ae6364"
        "b17b723cc3dfd0d74b975e17cff5bcb5"
    ),
    "global_mpnn_features": (
        "d4f474817c32c41e9ff20ff6377bff7"
        "a4aa45463ca218380968e445d6c31cdbc"
    ),
    "global_feature_row_manifest": (
        "56cf4d4009797180057b2516e53ac0806"
        "a69757c9943a7f52a4dcf9032c5775f"
    ),
}

INPUT_AUDIT_PATH = (
    LOG_DIR
    / "final_two_model_input_audit.csv"
)

INPUT_MANIFEST_PATH = (
    LOG_DIR
    / "final_two_model_input_manifest.json"
)

INTERFACE_AUDIT_PATH = (
    LOG_DIR
    / "final_two_model_training_interface_audit.json"
)

TUNING_SIGNATURE_PATH = (
    LOG_DIR
    / "final_two_model_tuning_signature.json"
)

STUDY_PLAN_PATH = (
    RESULT_DIR
    / "final_two_model_study_plan.csv"
)

TUNING_SUMMARY_PATH = (
    RESULT_DIR
    / "final_two_model_tuning_summary.csv"
)

TUNING_MANIFEST_PATH = (
    LOG_DIR
    / "final_two_model_tuning_manifest.json"
)

FINAL_SCALER_PATH = (
    SCALER_DIR
    / "final_downstream_feature_scaler.joblib"
)

FULL_INPUT_AUDIT_CSV_PATH = (
    LOG_DIR
    / "full_development_training_input_audit.csv"
)

FULL_INPUT_AUDIT_JSON_PATH = (
    LOG_DIR
    / "full_development_training_input_audit.json"
)

FINAL_LGBM_PATH = (
    MODEL_DIR
    / "final_LightGBM_viscosity.joblib"
)

FINAL_TRANSFORMER_PATH = (
    MODEL_DIR
    / "final_Transformer_viscosity.pt"
)

FINAL_TRANSFORMER_HISTORY_PATH = (
    RESULT_DIR
    / "final_Transformer_training_history.csv"
)

FINAL_TRAINING_SUMMARY_PATH = (
    RESULT_DIR
    / "final_model_training_summary.csv"
)

FINAL_REGISTRY_PATH = (
    RESULT_DIR
    / "final_two_model_registry.csv"
)

OUTPUT_CHECK_PATH = (
    LOG_DIR
    / "final_two_model_output_check.csv"
)

FINAL_FREEZE_MANIFEST_PATH = (
    LOG_DIR
    / "final_two_model_freeze_manifest.json"
)

OPTUNA_DB_PATH = (
    STUDY_DIR
    / "optuna_final_two_model_IL_split_5fold.db"
)

required_base_paths = [
    ML_READY_PATH,
    GLOBAL_FEATURE_PATH,
    GLOBAL_FEATURE_ROW_MANIFEST_PATH,
    MPNN_MODEL_PATH,
    MPNN_TP_SCALER_PATH,
    MPNN_TARGET_SCALER_PATH,
    MPNN_METADATA_PATH,
    PARAMETER_HANDOFF_MANIFEST_PATH,
    FORMAL_FINAL_MANIFEST_PATH,
    FOLDWISE_FEATURE_ROOT,
]

missing_base_paths = [
    path
    for path in required_base_paths
    if not Path(path).exists()
]

if missing_base_paths:
    raise FileNotFoundError(
        "缺少基础输入：\n"
        + "\n".join(
            map(
                str,
                missing_base_paths,
            )
        )
    )

seed_everything(42)

print("=" * 108)
print(
    "Cell 0 两模型最终参数确定与"
    "模型冻结环境设置完成 ✓"
)
print("=" * 108)

print(
    "python                         =",
    sys.executable,
)

print(
    "optuna                        =",
    optuna.__version__,
)

print(
    "lightgbm                      =",
    lightgbm.__version__,
)

print(
    "torch                         =",
    torch.__version__,
)

print(
    "device                        =",
    DEVICE,
)

print(
    "PROJECT_ROOT                  =",
    PROJECT_ROOT,
)

print(
    "EXPERIMENT_ROOT               =",
    EXPERIMENT_ROOT,
)

print(
    "PIPELINE_ROOT                 =",
    PIPELINE_ROOT,
)

print(
    "parameter handoff manifest    =",
    PARAMETER_HANDOFF_MANIFEST_PATH,
)

print(
    "formal final manifest         =",
    FORMAL_FINAL_MANIFEST_PATH,
)

print(
    "FINAL_ROOT                    =",
    FINAL_ROOT,
)

print(
    "models                        =",
    MODELS,
)

print(
    "final application model       =",
    FINAL_APPLICATION_MODEL,
)

print(
    "supplementary control model   =",
    SUPPLEMENTARY_CONTROL_MODEL,
)

print(
    "final parameter split logic   =",
    FINAL_SELECTION_SPLIT,
)

print(
    "trial budgets                 =",
    N_TRIALS,
)

print(
    "external 10-point file read   =",
    False,
)

Cell 0 两模型最终参数确定与模型冻结环境设置完成 ✓
python                         = D:\conda\envs\rdkit-env\python.exe
optuna                        = 4.8.0
lightgbm                      = 4.6.0
torch                         = 2.2.1+cpu
device                        = cpu
PROJECT_ROOT                  = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT               = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
PIPELINE_ROOT                 = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits
parameter handoff manifest    = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\hyperparameter_tuning_inner_foldwise_mpnn\summaries\4eafa9d6fbe7\outer_evaluation_handoff\outer_evaluation_parameter_manifest_Data_IL.json
formal final manifest         = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2

In [2]:
# Cell 1 | 审计16216条开发数据、全局MPNN表征和5套IL_split折输入

required_identity_columns = [
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
]

ml_identity = read_existing_columns(
    ML_READY_PATH,
    desired_columns=required_identity_columns,
    required_columns=required_identity_columns,
).reset_index(drop=True)

if len(ml_identity) != EXPECTED_ROWS:
    raise RuntimeError(
        f"ml_ready_A行数为{len(ml_identity)}，"
        f"预期为{EXPECTED_ROWS}。"
    )

if ml_identity.isna().any().any():
    raise RuntimeError(
        "开发数据身份字段存在缺失值。"
    )

actual_hashes = {
    "ml_ready_A": (
        calculate_sha256(
            ML_READY_PATH
        )
    ),
    "global_mpnn_features": (
        calculate_sha256(
            GLOBAL_FEATURE_PATH
        )
    ),
    "global_feature_row_manifest": (
        calculate_sha256(
            GLOBAL_FEATURE_ROW_MANIFEST_PATH
        )
    ),
}

for (
    role,
    expected_hash,
) in EXPECTED_HASHES.items():

    if (
        actual_hashes[role]
        != expected_hash
    ):
        raise RuntimeError(
            f"{role}的SHA256与固定开发数据版本不一致：\n"
            f"actual={actual_hashes[role]}\n"
            f"expected={expected_hash}"
        )

global_feature_memmap = np.load(
    GLOBAL_FEATURE_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

if (
    global_feature_memmap.shape
    != (
        EXPECTED_ROWS,
        EXPECTED_FEATURE_DIM,
    )
):
    raise RuntimeError(
        "全局MPNN特征形状错误："
        f"{global_feature_memmap.shape}"
    )

if not all_finite_memmap(
    GLOBAL_FEATURE_PATH
):
    raise RuntimeError(
        "全局MPNN特征中存在NaN或无穷值。"
    )

desired_manifest_columns = [
    "source_row_idx",
    "global_row_index",
    "feature_row",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
]

feature_row_manifest = (
    read_existing_columns(
        GLOBAL_FEATURE_ROW_MANIFEST_PATH,
        desired_columns=(
            desired_manifest_columns
        ),
    )
    .reset_index(
        drop=True
    )
)

if (
    len(feature_row_manifest)
    != EXPECTED_ROWS
):
    raise RuntimeError(
        "全局MPNN特征行清单行数不是16216。"
    )

for column in required_identity_columns:

    if column in feature_row_manifest.columns:

        if column in {
            "T_rep",
            "P_rep",
        }:
            left = pd.to_numeric(
                ml_identity[column],
                errors="raise",
            ).to_numpy(
                dtype=float
            )

            right = pd.to_numeric(
                feature_row_manifest[column],
                errors="raise",
            ).to_numpy(
                dtype=float
            )

            matched = np.allclose(
                left,
                right,
                atol=1e-12,
                rtol=0.0,
            )

        else:
            left = (
                ml_identity[column]
                .astype(str)
                .to_numpy()
            )

            right = (
                feature_row_manifest[column]
                .astype(str)
                .to_numpy()
            )

            matched = np.array_equal(
                left,
                right,
            )

        if not matched:
            raise RuntimeError(
                "全局MPNN特征行清单与"
                f"ml_ready_A的{column}顺序不一致。"
            )

mpnn_metadata = read_json(
    MPNN_METADATA_PATH
)

external_holdout_meta = (
    mpnn_metadata.get(
        "external_holdout",
        {},
    )
)

if (
    str(
        mpnn_metadata.get(
            "representation_method",
            "",
        )
    ).upper()
    != "MPNN"
):
    raise RuntimeError(
        "MPNN元数据中的表征方法不是MPNN。"
    )

if (
    int(
        mpnn_metadata.get(
            "n_total_rows",
            -1,
        )
    )
    != EXPECTED_ROWS
):
    raise RuntimeError(
        "MPNN元数据中的开发数据行数不是16216。"
    )

if (
    int(
        mpnn_metadata
        .get(
            "architecture",
            {},
        )
        .get(
            "descriptor_dim",
            -1,
        )
    )
    != EXPECTED_FEATURE_DIM
):
    raise RuntimeError(
        "MPNN元数据中的描述符维度不是130。"
    )

if bool(
    external_holdout_meta.get(
        "used_in_pretraining",
        True,
    )
):
    raise RuntimeError(
        "MPNN元数据表明外部10点参与了"
        "特征提取器预训练。"
    )

if (
    mpnn_metadata
    .get(
        "outputs",
        {},
    )
    .get(
        "feature_file_sha256"
    )
    != actual_hashes[
        "global_mpnn_features"
    ]
):
    raise RuntimeError(
        "MPNN元数据中的全局特征SHA256"
        "与当前文件不一致。"
    )

formal_manifest = read_json(
    FORMAL_FINAL_MANIFEST_PATH
)

if (
    formal_manifest.get(
        "status"
    )
    != (
        "formal_outer_evaluation_and_"
        "difficult_fold_diagnosis_finalized"
    )
):
    raise RuntimeError(
        "正式外层评价最终manifest状态不正确。"
    )

if (
    formal_manifest
    .get(
        "model_selection",
        {},
    )
    .get(
        "selected_final_application_model"
    )
    != "LightGBM"
):
    raise RuntimeError(
        "正式外层评价没有将LightGBM"
        "记录为最终应用模型。"
    )

if bool(
    formal_manifest
    .get(
        "model_selection",
        {},
    )
    .get(
        "cross_split_metric_averaging_used",
        True,
    )
):
    raise RuntimeError(
        "正式模型选择记录显示"
        "Data_split与IL_split被合并。"
    )

if bool(
    formal_manifest
    .get(
        "verification",
        {},
    )
    .get(
        "external_10point_holdout_read",
        True,
    )
):
    raise RuntimeError(
        "正式外层评价记录显示读取了外部10点。"
    )

parameter_handoff_manifest = (
    read_json(
        PARAMETER_HANDOFF_MANIFEST_PATH
    )
)

if (
    parameter_handoff_manifest.get(
        "status"
    )
    != "outer_evaluation_parameter_handoff_ready"
):
    raise RuntimeError(
        "参数交接manifest状态不正确。"
    )

handoff_design = (
    parameter_handoff_manifest.get(
        "handoff_design",
        {},
    )
)

if bool(
    handoff_design.get(
        "cross_outer_fold_parameter_pooling",
        True,
    )
):
    raise RuntimeError(
        "参数交接记录显示跨外层折合并了参数。"
    )

if bool(
    handoff_design.get(
        "global_parameter_averaging",
        True,
    )
):
    raise RuntimeError(
        "参数交接记录显示对外层折参数求了平均。"
    )

fold_audit_records = []
FOLD_REGISTRY = {}
all_test_indices = []

for fold_number in range(
    1,
    N_FOLDS + 1,
):
    fold_dir = (
        FOLDWISE_FEATURE_ROOT
        / f"Fold_{fold_number}"
    )

    paths = {
        "X_train": (
            fold_dir
            / "X_outer_train_mpnn.npy"
        ),
        "X_valid": (
            fold_dir
            / "X_outer_test_mpnn.npy"
        ),
        "y_train": (
            fold_dir
            / "y_outer_train_lg_eta.npy"
        ),
        "y_valid": (
            fold_dir
            / "y_outer_test_lg_eta.npy"
        ),
        "indices": (
            fold_dir
            / "fold_indices.npz"
        ),
        "train_manifest": (
            fold_dir
            / "outer_train_manifest.csv"
        ),
        "valid_manifest": (
            fold_dir
            / "outer_test_manifest.csv"
        ),
        "metadata": (
            fold_dir
            / "fold_feature_metadata.json"
        ),
        "audit": (
            fold_dir
            / "fold_output_audit.json"
        ),
    }

    missing = [
        path
        for path in paths.values()
        if not path.exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"IL_split Fold {fold_number}缺少文件：\n"
            + "\n".join(
                map(
                    str,
                    missing,
                )
            )
        )

    metadata = read_json(
        paths["metadata"]
    )

    output_audit = read_json(
        paths["audit"]
    )

    if (
        metadata.get(
            "split_name"
        )
        != "IL_split"
        or int(
            metadata.get(
                "fold",
                -1,
            )
        )
        != fold_number
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "元数据身份不一致。"
        )

    if (
        output_audit.get(
            "audit_status"
        )
        != "passed"
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "输出审计未通过。"
        )

    if bool(
        metadata
        .get(
            "external_holdout",
            {},
        )
        .get(
            "used_in_current_fold",
            True,
        )
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "使用了外部10点。"
        )

    leakage = metadata.get(
        "leakage_control",
        {},
    )

    if bool(
        leakage.get(
            "outer_test_used_for_mpnn_training",
            True,
        )
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "外层测试数据参与了MPNN训练。"
        )

    if bool(
        leakage.get(
            "outer_test_used_for_early_stopping",
            True,
        )
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "外层测试数据参与了MPNN早停。"
        )

    with np.load(
        paths["indices"],
        allow_pickle=False,
    ) as index_bundle:

        required_keys = {
            "outer_train_idx",
            "outer_test_idx",
        }

        if not required_keys.issubset(
            index_bundle.files
        ):
            raise KeyError(
                f"IL_split Fold {fold_number}"
                f"索引文件缺少{required_keys}。"
            )

        train_idx = np.asarray(
            index_bundle[
                "outer_train_idx"
            ],
            dtype=int,
        )

        valid_idx = np.asarray(
            index_bundle[
                "outer_test_idx"
            ],
            dtype=int,
        )

    if (
        len(
            np.intersect1d(
                train_idx,
                valid_idx,
            )
        )
        != 0
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "训练与验证索引重叠。"
        )

    if (
        len(
            np.union1d(
                train_idx,
                valid_idx,
            )
        )
        != EXPECTED_ROWS
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "训练与验证索引未覆盖全部开发数据。"
        )

    if (
        train_idx.min() < 0
        or valid_idx.min() < 0
        or train_idx.max() >= EXPECTED_ROWS
        or valid_idx.max() >= EXPECTED_ROWS
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "索引越界。"
        )

    train_groups = set(
        ml_identity.iloc[
            train_idx
        ][
            "IL_key"
        ].astype(str)
    )

    valid_groups = set(
        ml_identity.iloc[
            valid_idx
        ][
            "IL_key"
        ].astype(str)
    )

    group_overlap = len(
        train_groups
        & valid_groups
    )

    if group_overlap != 0:
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "训练与验证IL_key重叠："
            f"{group_overlap}"
        )

    X_train = np.load(
        paths["X_train"],
        mmap_mode="r",
        allow_pickle=False,
    )

    X_valid = np.load(
        paths["X_valid"],
        mmap_mode="r",
        allow_pickle=False,
    )

    y_train = np.load(
        paths["y_train"],
        mmap_mode="r",
        allow_pickle=False,
    )

    if (
        X_train.shape
        != (
            len(train_idx),
            EXPECTED_FEATURE_DIM,
        )
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "训练特征形状错误："
            f"{X_train.shape}"
        )

    if (
        X_valid.shape
        != (
            len(valid_idx),
            EXPECTED_FEATURE_DIM,
        )
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "验证特征形状错误："
            f"{X_valid.shape}"
        )

    if (
        np.asarray(
            y_train
        )
        .reshape(-1)
        .shape
        != (
            len(train_idx),
        )
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "训练目标形状错误。"
        )

    if (
        not all_finite_memmap(
            paths["X_train"]
        )
        or not all_finite_memmap(
            paths["X_valid"]
        )
    ):
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "特征中存在NaN或无穷值。"
        )

    if not np.isfinite(
        np.asarray(
            y_train
        )
    ).all():
        raise RuntimeError(
            f"IL_split Fold {fold_number}"
            "训练目标中存在NaN或无穷值。"
        )

    # y_valid仅记录路径，本Cell不打开、不计算哈希。
    FOLD_REGISTRY[
        fold_number
    ] = {
        **paths,
        "train_idx": train_idx,
        "valid_idx": valid_idx,
    }

    all_test_indices.extend(
        valid_idx.tolist()
    )

    fold_audit_records.append(
        {
            "Split": "IL_split",
            "Fold": fold_number,
            "Train_rows": len(
                train_idx
            ),
            "Validation_rows": len(
                valid_idx
            ),
            "Feature_dim": (
                EXPECTED_FEATURE_DIM
            ),
            "Train_validation_index_overlap": 0,
            "IL_key_overlap": group_overlap,
            "Fold_output_audit_status": (
                output_audit.get(
                    "audit_status"
                )
            ),
            "External_10point_used": False,
            "Validation_target_opened_in_Cell_1": False,
            "Status": "passed",
        }
    )

all_test_indices = np.asarray(
    all_test_indices,
    dtype=int,
)

unique_indices, counts = np.unique(
    all_test_indices,
    return_counts=True,
)

if (
    len(unique_indices)
    != EXPECTED_ROWS
    or not np.all(
        counts == 1
    )
):
    raise RuntimeError(
        "5个IL_split验证折没有恰好覆盖"
        "全部16216条开发数据一次。"
    )

input_audit_df = pd.DataFrame(
    fold_audit_records
)

save_csv_atomic(
    input_audit_df,
    INPUT_AUDIT_PATH,
)

input_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "run_code_version": (
        RUN_CODE_VERSION
    ),
    "status": (
        "final_two_model_inputs_verified"
    ),
    "design": {
        "models": MODELS,
        "final_application_model": (
            FINAL_APPLICATION_MODEL
        ),
        "supplementary_control_model": (
            SUPPLEMENTARY_CONTROL_MODEL
        ),
        "parameter_selection_split": (
            FINAL_SELECTION_SPLIT
        ),
        "development_rows": (
            EXPECTED_ROWS
        ),
        "IL_split_folds": (
            N_FOLDS
        ),
        "external_10point_read": False,
        "fold_validation_targets_opened": False,
    },
    "global_inputs": {
        "ml_ready_A": str(
            ML_READY_PATH
        ),
        "global_mpnn_features": str(
            GLOBAL_FEATURE_PATH
        ),
        "global_feature_row_manifest": str(
            GLOBAL_FEATURE_ROW_MANIFEST_PATH
        ),
        "mpnn_model": str(
            MPNN_MODEL_PATH
        ),
        "mpnn_tp_scaler": str(
            MPNN_TP_SCALER_PATH
        ),
        "mpnn_target_scaler": str(
            MPNN_TARGET_SCALER_PATH
        ),
        "mpnn_metadata": str(
            MPNN_METADATA_PATH
        ),
        "formal_final_manifest": str(
            FORMAL_FINAL_MANIFEST_PATH
        ),
        "parameter_handoff_manifest": str(
            PARAMETER_HANDOFF_MANIFEST_PATH
        ),
    },
    "global_input_sha256": {
        **actual_hashes,
        "mpnn_model": (
            calculate_sha256(
                MPNN_MODEL_PATH
            )
        ),
        "mpnn_tp_scaler": (
            calculate_sha256(
                MPNN_TP_SCALER_PATH
            )
        ),
        "mpnn_target_scaler": (
            calculate_sha256(
                MPNN_TARGET_SCALER_PATH
            )
        ),
        "mpnn_metadata": (
            calculate_sha256(
                MPNN_METADATA_PATH
            )
        ),
        "formal_final_manifest": (
            calculate_sha256(
                FORMAL_FINAL_MANIFEST_PATH
            )
        ),
        "parameter_handoff_manifest": (
            calculate_sha256(
                PARAMETER_HANDOFF_MANIFEST_PATH
            )
        ),
    },
    "alignment": {
        "global_feature_shape": list(
            global_feature_memmap.shape
        ),
        "feature_row_manifest_rows": len(
            feature_row_manifest
        ),
        "five_validation_folds_cover_each_"
        "development_row_once": True,
        "all_IL_key_overlaps_zero": True,
        "all_fold_feature_audits_passed": True,
        "external_10point_used_in_mpnn_"
        "pretraining": False,
    },
    "fold_records": (
        input_audit_df.to_dict(
            orient="records"
        )
    ),
    "outputs": {
        "input_audit": str(
            INPUT_AUDIT_PATH
        ),
        "input_manifest": str(
            INPUT_MANIFEST_PATH
        ),
    },
}

save_json_atomic(
    input_manifest,
    INPUT_MANIFEST_PATH,
)

print("=" * 108)
print(
    "Cell 1 开发数据、全局MPNN表征和"
    "5套IL_split折输入审计通过 ✓"
)
print("=" * 108)

print(
    "development rows                    =",
    len(ml_identity),
)

print(
    "global MPNN feature shape           =",
    global_feature_memmap.shape,
)

print(
    "IL_split folds checked              =",
    len(input_audit_df),
)

print(
    "all validation rows covered once    =",
    True,
)

print(
    "all IL_key overlaps                 =",
    0,
)

print(
    "validation targets opened           =",
    False,
)

print(
    "external 10-point file read         =",
    False,
)

print(
    "input audit                         =",
    INPUT_AUDIT_PATH,
)

print(
    "input manifest                      =",
    INPUT_MANIFEST_PATH,
)

display(
    input_audit_df
)

Cell 1 开发数据、全局MPNN表征和5套IL_split折输入审计通过 ✓
development rows                    = 16216
global MPNN feature shape           = (16216, 130)
IL_split folds checked              = 5
all validation rows covered once    = True
all IL_key overlaps                 = 0
validation targets opened           = False
external 10-point file read         = False
input audit                         = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\logs\final_two_model_input_audit.csv
input manifest                      = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\logs\final_two_model_input_manifest.json


,Split,Fold,Train_rows,Validation_rows,Feature_dim,Train_validation_index_overlap,IL_key_overlap,Fold_output_audit_status,External_10point_used,Validation_target_opened_in_Cell_1,Status
0,IL_split,1,12972,3244,130,0,0,passed,False,False,passed
1,IL_split,2,12973,3243,130,0,0,passed,False,False,passed
2,IL_split,3,12973,3243,130,0,0,passed,False,False,passed
3,IL_split,4,12973,3243,130,0,0,passed,False,False,passed
4,IL_split,5,12973,3243,130,0,0,passed,False,False,passed


In [3]:
# Cell 2 | 定义评价指标、LightGBM搜索空间和Transformer训练接口

METRIC_ORDER = [
    "R2",
    "RMSE",
    "MAE",
    "Pearson_r",
]


def safe_pearson_r(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=np.float64,
    ).reshape(-1)

    y_pred = np.asarray(
        y_pred,
        dtype=np.float64,
    ).reshape(-1)

    if (
        np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    return float(
        pearsonr(
            y_true,
            y_pred,
        )[0]
    )


def calculate_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=np.float64,
    ).reshape(-1)

    y_pred = np.asarray(
        y_pred,
        dtype=np.float64,
    ).reshape(-1)

    if (
        len(y_true) != len(y_pred)
        or len(y_true) < 2
    ):
        raise ValueError(
            "真实值与预测值长度不一致"
            "或样本数不足。"
        )

    if (
        not np.isfinite(
            y_true
        ).all()
        or not np.isfinite(
            y_pred
        ).all()
    ):
        raise ValueError(
            "评价指标输入中存在NaN或无穷值。"
        )

    return {
        "R2": float(
            r2_score(
                y_true,
                y_pred,
            )
        ),
        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred,
                )
            )
        ),
        "MAE": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "Pearson_r": (
            safe_pearson_r(
                y_true,
                y_pred,
            )
        ),
    }


def aggregate_fold_metrics(
    fold_records,
):
    if len(fold_records) != N_FOLDS:
        raise ValueError(
            "折指标记录数不是5。"
        )

    dataframe = pd.DataFrame(
        fold_records
    )

    result = {}

    for metric in METRIC_ORDER:
        values = pd.to_numeric(
            dataframe[metric],
            errors="raise",
        ).to_numpy(
            dtype=float
        )

        if not np.isfinite(
            values
        ).all():
            raise ValueError(
                f"{metric}中存在NaN或无穷值。"
            )

        result[
            f"Mean_{metric}"
        ] = float(
            np.mean(values)
        )

        result[
            f"Std_{metric}"
        ] = float(
            np.std(
                values,
                ddof=0,
            )
        )

    result[
        "Objective_value"
    ] = result[
        "Mean_R2"
    ]

    return result


def get_scaler_fitted_rows(
    scaler,
):
    value = np.asarray(
        scaler.n_samples_seen_
    ).reshape(-1)

    if not np.all(
        value == value[0]
    ):
        raise RuntimeError(
            "StandardScaler各特征记录的"
            "拟合行数不一致。"
        )

    return int(
        value[0]
    )


def suggest_lightgbm_parameters(
    trial,
):
    return {
        "n_estimators": (
            trial.suggest_int(
                "n_estimators",
                200,
                1000,
                step=50,
            )
        ),
        "learning_rate": (
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.15,
                log=True,
            )
        ),
        "max_depth": (
            trial.suggest_categorical(
                "max_depth",
                [
                    -1,
                    6,
                    8,
                    10,
                    12,
                    16,
                ],
            )
        ),
        "num_leaves": (
            trial.suggest_int(
                "num_leaves",
                15,
                127,
                step=4,
            )
        ),
        "min_child_samples": (
            trial.suggest_int(
                "min_child_samples",
                5,
                60,
                step=5,
            )
        ),
        "subsample": (
            trial.suggest_float(
                "subsample",
                0.65,
                1.00,
            )
        ),
        "colsample_bytree": (
            trial.suggest_float(
                "colsample_bytree",
                0.60,
                1.00,
            )
        ),
        "reg_alpha": (
            trial.suggest_float(
                "reg_alpha",
                1e-8,
                2.0,
                log=True,
            )
        ),
        "reg_lambda": (
            trial.suggest_float(
                "reg_lambda",
                1e-8,
                5.0,
                log=True,
            )
        ),
        "min_split_gain": (
            trial.suggest_float(
                "min_split_gain",
                0.0,
                0.20,
            )
        ),
    }


def lightgbm_search_to_formal(
    search_parameters,
    seed,
):
    return {
        "n_estimators": int(
            search_parameters[
                "n_estimators"
            ]
        ),
        "learning_rate": float(
            search_parameters[
                "learning_rate"
            ]
        ),
        "max_depth": int(
            search_parameters[
                "max_depth"
            ]
        ),
        "num_leaves": int(
            search_parameters[
                "num_leaves"
            ]
        ),
        "min_child_samples": int(
            search_parameters[
                "min_child_samples"
            ]
        ),
        "subsample": float(
            search_parameters[
                "subsample"
            ]
        ),
        "subsample_freq": 1,
        "colsample_bytree": float(
            search_parameters[
                "colsample_bytree"
            ]
        ),
        "reg_alpha": float(
            search_parameters[
                "reg_alpha"
            ]
        ),
        "reg_lambda": float(
            search_parameters[
                "reg_lambda"
            ]
        ),
        "min_split_gain": float(
            search_parameters[
                "min_split_gain"
            ]
        ),
        "objective": "regression",
        "random_state": int(seed),
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    }


class OptunaAttentionBlock(
    nn.Module
):
    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout,
    ):
        super().__init__()

        if (
            embed_dim
            % num_heads
            != 0
        ):
            raise ValueError(
                "embed_dim必须能被num_heads整除。"
            )

        self.attention = (
            nn.MultiheadAttention(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.norm1 = nn.LayerNorm(
            embed_dim
        )

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        self.feed_forward = (
            nn.Sequential(
                nn.Linear(
                    embed_dim,
                    embed_dim * 4,
                ),
                nn.GELU(),
                nn.Dropout(
                    dropout
                ),
                nn.Linear(
                    embed_dim * 4,
                    embed_dim,
                ),
            )
        )

    def forward(
        self,
        x,
    ):
        attention_output, _ = (
            self.attention(
                x,
                x,
                x,
                need_weights=False,
            )
        )

        x = self.norm1(
            x
            + attention_output
        )

        return self.norm2(
            x
            + self.feed_forward(
                x
            )
        )


class OptunaTransformerRegressor(
    nn.Module
):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_tokens,
        num_heads,
        num_layers,
        dropout,
        noise_std=0.01,
    ):
        super().__init__()

        if (
            hidden_dim
            % num_tokens
            != 0
        ):
            raise ValueError(
                "hidden_dim必须能被num_tokens整除。"
            )

        token_dim = (
            hidden_dim
            // num_tokens
        )

        if (
            token_dim
            % num_heads
            != 0
        ):
            raise ValueError(
                "token_dim必须能被num_heads整除。"
            )

        self.input_dim = int(
            input_dim
        )

        self.num_tokens = int(
            num_tokens
        )

        self.token_dim = int(
            token_dim
        )

        self.noise_std = float(
            noise_std
        )

        self.input_projection = (
            nn.Linear(
                input_dim,
                hidden_dim,
            )
        )

        self.blocks = nn.ModuleList(
            [
                OptunaAttentionBlock(
                    token_dim,
                    num_heads,
                    dropout,
                )
                for _ in range(
                    num_layers
                )
            ]
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                hidden_dim,
                64,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                64,
                1,
            ),
        )

    def forward(
        self,
        x,
    ):
        if (
            x.ndim != 2
            or x.shape[1]
            != self.input_dim
        ):
            raise ValueError(
                "Transformer输入形状错误。"
            )

        if (
            self.training
            and self.noise_std > 0
        ):
            x = (
                x
                + torch.randn_like(
                    x
                )
                * self.noise_std
            )

        x = self.input_projection(
            x
        )

        x = x.reshape(
            x.shape[0],
            self.num_tokens,
            self.token_dim,
        )

        for block in self.blocks:
            x = block(x)

        return self.regressor(
            x
        )


def suggest_transformer_parameters(
    trial,
):
    return {
        "hidden_dim": (
            trial.suggest_categorical(
                "hidden_dim",
                [
                    64,
                    128,
                ],
            )
        ),
        "num_tokens": (
            TRANSFORMER_NUM_TOKENS
        ),
        "num_heads": (
            trial.suggest_categorical(
                "num_heads",
                [
                    1,
                    2,
                    4,
                ],
            )
        ),
        "num_layers": (
            trial.suggest_int(
                "num_layers",
                1,
                4,
            )
        ),
        "lr": (
            trial.suggest_float(
                "lr",
                5e-4,
                5e-3,
                log=True,
            )
        ),
        "dropout": (
            trial.suggest_float(
                "dropout",
                0.20,
                0.60,
            )
        ),
        "weight_decay": (
            trial.suggest_float(
                "weight_decay",
                1e-4,
                5e-2,
                log=True,
            )
        ),
        "batch_size": (
            trial.suggest_categorical(
                "batch_size",
                [
                    32,
                    64,
                    128,
                ],
            )
        ),
        "noise_std": (
            TRANSFORMER_NOISE_STD
        ),
    }


def transformer_search_to_formal(
    search_parameters,
    seed,
):
    return {
        "input_dim": (
            EXPECTED_FEATURE_DIM
        ),
        "hidden_dim": int(
            search_parameters[
                "hidden_dim"
            ]
        ),
        "num_tokens": int(
            search_parameters.get(
                "num_tokens",
                TRANSFORMER_NUM_TOKENS,
            )
        ),
        "num_heads": int(
            search_parameters[
                "num_heads"
            ]
        ),
        "num_layers": int(
            search_parameters[
                "num_layers"
            ]
        ),
        "lr": float(
            search_parameters[
                "lr"
            ]
        ),
        "dropout": float(
            search_parameters[
                "dropout"
            ]
        ),
        "weight_decay": float(
            search_parameters[
                "weight_decay"
            ]
        ),
        "batch_size": int(
            search_parameters[
                "batch_size"
            ]
        ),
        "noise_std": float(
            search_parameters.get(
                "noise_std",
                TRANSFORMER_NOISE_STD,
            )
        ),
        "search_epochs": (
            TRANSFORMER_SEARCH_EPOCHS
        ),
        "formal_epochs": (
            TRANSFORMER_FORMAL_EPOCHS
        ),
        "seed": int(seed),
    }


def build_transformer_model(
    parameters,
):
    return OptunaTransformerRegressor(
        input_dim=int(
            parameters.get(
                "input_dim",
                EXPECTED_FEATURE_DIM,
            )
        ),
        hidden_dim=int(
            parameters[
                "hidden_dim"
            ]
        ),
        num_tokens=int(
            parameters.get(
                "num_tokens",
                TRANSFORMER_NUM_TOKENS,
            )
        ),
        num_heads=int(
            parameters[
                "num_heads"
            ]
        ),
        num_layers=int(
            parameters[
                "num_layers"
            ]
        ),
        dropout=float(
            parameters[
                "dropout"
            ]
        ),
        noise_std=float(
            parameters.get(
                "noise_std",
                TRANSFORMER_NOISE_STD,
            )
        ),
    )


def predict_transformer_in_batches(
    model,
    X,
    device,
    batch_size=1024,
):
    X = np.asarray(
        X,
        dtype=np.float32,
    )

    dataset = TensorDataset(
        torch.from_numpy(
            X
        )
    )

    loader = DataLoader(
        dataset,
        batch_size=int(
            batch_size
        ),
        shuffle=False,
        num_workers=0,
    )

    parts = []

    model.eval()

    with torch.no_grad():
        for (
            batch_X,
        ) in loader:

            parts.append(
                model(
                    batch_X.to(
                        device
                    )
                )
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

    prediction = np.concatenate(
        parts
    ).astype(
        np.float32
    )

    if (
        len(prediction)
        != len(X)
        or not np.isfinite(
            prediction
        ).all()
    ):
        raise RuntimeError(
            "Transformer预测结果无效。"
        )

    return prediction


def fit_transformer_core(
    X_train,
    y_train,
    parameters,
    device,
    epochs,
    seed,
):
    seed_everything(
        int(seed)
    )

    X_train = np.asarray(
        X_train,
        dtype=np.float32,
    )

    y_train = np.asarray(
        y_train,
        dtype=np.float32,
    ).reshape(
        -1,
        1,
    )

    model = build_transformer_model(
        parameters
    ).to(
        device
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=float(
            parameters[
                "lr"
            ]
        ),
        weight_decay=float(
            parameters[
                "weight_decay"
            ]
        ),
    )

    criterion = nn.MSELoss()

    dataset = TensorDataset(
        torch.from_numpy(
            X_train
        ),
        torch.from_numpy(
            y_train
        ),
    )

    generator = (
        torch.Generator()
        .manual_seed(
            int(seed)
        )
    )

    loader = DataLoader(
        dataset,
        batch_size=int(
            parameters[
                "batch_size"
            ]
        ),
        shuffle=True,
        num_workers=0,
        generator=generator,
        drop_last=False,
    )

    scheduler = OneCycleLR(
        optimizer,
        max_lr=float(
            parameters[
                "lr"
            ]
        ),
        steps_per_epoch=len(
            loader
        ),
        epochs=int(
            epochs
        ),
        pct_start=0.10,
        anneal_strategy="cos",
    )

    history = []

    for epoch in range(
        1,
        int(epochs) + 1,
    ):
        model.train()

        loss_sum = 0.0
        rows = 0

        for (
            batch_X,
            batch_y,
        ) in loader:

            batch_X = batch_X.to(
                device
            )

            batch_y = batch_y.to(
                device
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            loss = criterion(
                model(
                    batch_X
                ),
                batch_y,
            )

            if not torch.isfinite(
                loss
            ):
                raise RuntimeError(
                    "Transformer训练损失出现"
                    "NaN或无穷值。"
                )

            loss.backward()

            gradient_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )
            )

            if not torch.isfinite(
                gradient_norm
            ):
                raise RuntimeError(
                    "Transformer梯度范数出现"
                    "NaN或无穷值。"
                )

            optimizer.step()
            scheduler.step()

            batch_rows = len(
                batch_y
            )

            loss_sum += (
                float(
                    loss.item()
                )
                * batch_rows
            )

            rows += batch_rows

        history.append(
            {
                "Epoch": epoch,
                "Train_MSE": (
                    loss_sum
                    / rows
                ),
                "Learning_rate": float(
                    optimizer
                    .param_groups[0][
                        "lr"
                    ]
                ),
                "Training_rows_used": rows,
                "Validation_rows_used_for_"
                "backpropagation": 0,
            }
        )

    return (
        model,
        history,
    )


def train_predict_transformer_fold(
    X_train,
    y_train,
    X_validation,
    parameters,
    device,
    seed,
):
    model, history = (
        fit_transformer_core(
            X_train=X_train,
            y_train=y_train,
            parameters=parameters,
            device=device,
            epochs=(
                TRANSFORMER_SEARCH_EPOCHS
            ),
            seed=seed,
        )
    )

    prediction = (
        predict_transformer_in_batches(
            model,
            X_validation,
            device,
        )
    )

    final_train_mse = float(
        history[-1][
            "Train_MSE"
        ]
    )

    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        prediction,
        final_train_mse,
    )


dummy_parameters = {
    "input_dim": 130,
    "hidden_dim": 64,
    "num_tokens": 8,
    "num_heads": 1,
    "num_layers": 1,
    "dropout": 0.2,
    "noise_std": 0.01,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 32,
}

dummy_model = build_transformer_model(
    dummy_parameters
)

dummy_output = dummy_model(
    torch.zeros(
        (
            2,
            130,
        ),
        dtype=torch.float32,
    )
)

if (
    tuple(
        dummy_output.shape
    )
    != (
        2,
        1,
    )
):
    raise RuntimeError(
        "Transformer前向接口测试失败。"
    )

interface_audit = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "run_code_version": (
        RUN_CODE_VERSION
    ),
    "status": (
        "final_two_model_training_"
        "interfaces_verified"
    ),
    "lightgbm": {
        "search_space_matches_strict_"
        "tuning_notebook": True,
        "validation_target_argument_exists": False,
        "training_scaler_fitted_on_"
        "training_fold_only": True,
    },
    "transformer": {
        "architecture_matches_strict_"
        "tuning_notebook": True,
        "optimizer": "AdamW",
        "loss": "MSELoss",
        "scheduler": "OneCycleLR",
        "scheduler_pct_start": 0.10,
        "gradient_clip_max_norm": 1.0,
        "search_epochs": (
            TRANSFORMER_SEARCH_EPOCHS
        ),
        "formal_epochs": (
            TRANSFORMER_FORMAL_EPOCHS
        ),
        "train_fold_function_arguments": list(
            inspect.signature(
                train_predict_transformer_fold
            ).parameters
        ),
        "validation_target_argument_exists": False,
        "dummy_forward_shape": list(
            dummy_output.shape
        ),
    },
    "external_10point_read": False,
}

save_json_atomic(
    interface_audit,
    INTERFACE_AUDIT_PATH,
)

del (
    dummy_model,
    dummy_output,
)

print("=" * 108)
print(
    "Cell 2 两模型搜索空间和"
    "训练接口定义完成 ✓"
)
print("=" * 108)

print(
    "LightGBM trial budget              =",
    N_TRIALS[
        "LightGBM"
    ],
)

print(
    "Transformer trial budget           =",
    N_TRIALS[
        "Transformer"
    ],
)

print(
    "Transformer search epochs          =",
    TRANSFORMER_SEARCH_EPOCHS,
)

print(
    "Transformer formal epochs          =",
    TRANSFORMER_FORMAL_EPOCHS,
)

print(
    "Transformer validation target arg  =",
    False,
)

print(
    "external 10-point file read        =",
    False,
)

print(
    "interface audit                    =",
    INTERFACE_AUDIT_PATH,
)

Cell 2 两模型搜索空间和训练接口定义完成 ✓
LightGBM trial budget              = 20
Transformer trial budget           = 10
Transformer search epochs          = 50
Transformer formal epochs          = 150
Transformer validation target arg  = False
external 10-point file read        = False
interface audit                    = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\logs\final_two_model_training_interface_audit.json


In [4]:
# Cell 3 | 构建5折延迟读取验证目标的最终调参接口和Study计划

def make_trial_seed(
    model_name,
    trial_number,
    fold_number,
):
    model_offset = {
        "LightGBM": 20_000,
        "Transformer": 30_000,
    }[
        model_name
    ]

    return int(
        TUNING_RANDOM_SEED
        + model_offset
        + int(
            trial_number
        )
        * 100
        + int(
            fold_number
        )
    )


def load_fold_before_prediction(
    fold_number,
):
    registry = FOLD_REGISTRY[
        int(
            fold_number
        )
    ]

    X_train_raw = np.load(
        registry[
            "X_train"
        ],
        allow_pickle=False,
    ).astype(
        np.float32
    )

    X_validation_raw = np.load(
        registry[
            "X_valid"
        ],
        allow_pickle=False,
    ).astype(
        np.float32
    )

    y_train = np.load(
        registry[
            "y_train"
        ],
        allow_pickle=False,
    ).astype(
        np.float32
    ).reshape(-1)

    scaler = StandardScaler()

    X_train = (
        scaler
        .fit_transform(
            X_train_raw
        )
        .astype(
            np.float32
        )
    )

    X_validation = (
        scaler
        .transform(
            X_validation_raw
        )
        .astype(
            np.float32
        )
    )

    fitted_rows = (
        get_scaler_fitted_rows(
            scaler
        )
    )

    if fitted_rows != len(
        X_train
    ):
        raise RuntimeError(
            f"Fold {fold_number} "
            "StandardScaler拟合行数错误。"
        )

    return {
        "X_train": X_train,
        "X_validation": (
            X_validation
        ),
        "y_train": y_train,
        "y_validation_path": (
            registry[
                "y_valid"
            ]
        ),
        "train_rows": len(
            X_train
        ),
        "validation_rows": len(
            X_validation
        ),
        "scaler_fitted_rows": (
            fitted_rows
        ),
    }


def freeze_prediction_then_read_validation_target(
    prediction,
    validation_target_path,
):
    frozen_prediction = np.asarray(
        prediction,
        dtype=np.float32,
    ).reshape(-1).copy()

    frozen_prediction.setflags(
        write=False
    )

    prediction_hash_before = (
        array_sha256(
            frozen_prediction
        )
    )

    y_validation = np.load(
        validation_target_path,
        allow_pickle=False,
    ).astype(
        np.float32
    ).reshape(-1)

    prediction_hash_after = (
        array_sha256(
            frozen_prediction
        )
    )

    if (
        prediction_hash_before
        != prediction_hash_after
    ):
        raise RuntimeError(
            "读取验证目标后预测数组发生变化。"
        )

    if (
        len(frozen_prediction)
        != len(y_validation)
    ):
        raise RuntimeError(
            "验证预测与验证目标行数不一致。"
        )

    return (
        frozen_prediction,
        y_validation,
        prediction_hash_before,
    )


def suggest_model_parameters(
    model_name,
    trial,
):
    if model_name == "LightGBM":
        return suggest_lightgbm_parameters(
            trial
        )

    if model_name == "Transformer":
        return suggest_transformer_parameters(
            trial
        )

    raise KeyError(
        model_name
    )


def fit_predict_one_fold(
    model_name,
    search_parameters,
    prepared_fold,
    trial_number,
    fold_number,
):
    seed = make_trial_seed(
        model_name,
        trial_number,
        fold_number,
    )

    if model_name == "LightGBM":
        formal_parameters = (
            lightgbm_search_to_formal(
                search_parameters,
                seed,
            )
        )

        model = LGBMRegressor(
            **formal_parameters
        )

        model.fit(
            prepared_fold[
                "X_train"
            ],
            prepared_fold[
                "y_train"
            ],
        )

        prediction = model.predict(
            prepared_fold[
                "X_validation"
            ]
        )

        final_train_mse = None

        del model

    elif model_name == "Transformer":
        formal_parameters = (
            transformer_search_to_formal(
                search_parameters,
                seed,
            )
        )

        (
            prediction,
            final_train_mse,
        ) = (
            train_predict_transformer_fold(
                X_train=(
                    prepared_fold[
                        "X_train"
                    ]
                ),
                y_train=(
                    prepared_fold[
                        "y_train"
                    ]
                ),
                X_validation=(
                    prepared_fold[
                        "X_validation"
                    ]
                ),
                parameters=(
                    formal_parameters
                ),
                device=DEVICE,
                seed=seed,
            )
        )

    else:
        raise KeyError(
            model_name
        )

    prediction = np.asarray(
        prediction,
        dtype=np.float32,
    ).reshape(-1)

    if (
        len(prediction)
        != prepared_fold[
            "validation_rows"
        ]
        or not np.isfinite(
            prediction
        ).all()
    ):
        raise RuntimeError(
            "验证预测行数或数值无效。"
        )

    return (
        prediction,
        formal_parameters,
        seed,
        final_train_mse,
    )


def evaluate_final_trial(
    trial,
    model_name,
):
    search_parameters = (
        suggest_model_parameters(
            model_name,
            trial,
        )
    )

    fold_metric_records = []
    fold_audit_records = []

    trial_start = time.time()

    for fold_number in range(
        1,
        N_FOLDS + 1,
    ):
        prepared = (
            load_fold_before_prediction(
                fold_number
            )
        )

        (
            prediction,
            formal_parameters,
            seed,
            final_train_mse,
        ) = fit_predict_one_fold(
            model_name=(
                model_name
            ),
            search_parameters=(
                search_parameters
            ),
            prepared_fold=(
                prepared
            ),
            trial_number=(
                trial.number
            ),
            fold_number=(
                fold_number
            ),
        )

        (
            frozen_prediction,
            y_validation,
            prediction_hash,
        ) = (
            freeze_prediction_then_read_validation_target(
                prediction,
                prepared[
                    "y_validation_path"
                ],
            )
        )

        metrics = calculate_metrics(
            y_validation,
            frozen_prediction,
        )

        fold_metric_records.append(
            {
                "Fold": fold_number,
                **metrics,
            }
        )

        fold_audit_records.append(
            {
                "Fold": fold_number,
                "Train_rows": (
                    prepared[
                        "train_rows"
                    ]
                ),
                "Validation_rows": (
                    prepared[
                        "validation_rows"
                    ]
                ),
                "Scaler_fitted_rows": (
                    prepared[
                        "scaler_fitted_rows"
                    ]
                ),
                "Fold_seed": seed,
                "Prediction_sha256_before_"
                "target_read": (
                    prediction_hash
                ),
                "Validation_target_read_after_"
                "prediction_frozen": True,
                "Validation_target_used_for_"
                "training": False,
                "Validation_target_used_for_"
                "scaler_fit": False,
                "Validation_target_used_for_"
                "backpropagation": False,
                "Validation_target_used_for_"
                "early_stopping": False,
                "Validation_target_used_for_"
                "scheduler": False,
                "Final_train_MSE": (
                    final_train_mse
                ),
                **metrics,
            }
        )

        del (
            prepared,
            prediction,
            frozen_prediction,
            y_validation,
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    aggregate = aggregate_fold_metrics(
        fold_metric_records
    )

    for (
        key,
        value,
    ) in aggregate.items():
        trial.set_user_attr(
            key,
            float(value),
        )

    trial.set_user_attr(
        "Fold_records",
        to_jsonable(
            fold_audit_records
        ),
    )

    trial.set_user_attr(
        "Validation_target_policy",
        "read_after_prediction_frozen_only",
    )

    trial.set_user_attr(
        "External_10point_read",
        False,
    )

    trial_result_path = (
        TRIAL_DIR
        / (
            f"{model_name}_trial_"
            f"{trial.number:04d}.json"
        )
    )

    save_json_atomic(
        {
            "created_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),
            "model": model_name,
            "trial_number": (
                trial.number
            ),
            "search_parameters": (
                search_parameters
            ),
            "aggregate_metrics": (
                aggregate
            ),
            "fold_records": (
                fold_audit_records
            ),
            "elapsed_seconds": (
                time.time()
                - trial_start
            ),
            "external_10point_read": False,
        },
        trial_result_path,
    )

    return float(
        aggregate[
            "Objective_value"
        ]
    )


search_space_definition = {
    "LightGBM": {
        "n_estimators": [
            200,
            1000,
            50,
        ],
        "learning_rate": [
            0.01,
            0.15,
            "log",
        ],
        "max_depth": [
            -1,
            6,
            8,
            10,
            12,
            16,
        ],
        "num_leaves": [
            15,
            127,
            4,
        ],
        "min_child_samples": [
            5,
            60,
            5,
        ],
        "subsample": [
            0.65,
            1.00,
        ],
        "colsample_bytree": [
            0.60,
            1.00,
        ],
        "reg_alpha": [
            1e-8,
            2.0,
            "log",
        ],
        "reg_lambda": [
            1e-8,
            5.0,
            "log",
        ],
        "min_split_gain": [
            0.0,
            0.20,
        ],
    },
    "Transformer": {
        "hidden_dim": [
            64,
            128,
        ],
        "num_tokens": 8,
        "num_heads": [
            1,
            2,
            4,
        ],
        "num_layers": [
            1,
            4,
        ],
        "lr": [
            5e-4,
            5e-3,
            "log",
        ],
        "dropout": [
            0.20,
            0.60,
        ],
        "weight_decay": [
            1e-4,
            5e-2,
            "log",
        ],
        "batch_size": [
            32,
            64,
            128,
        ],
        "noise_std": 0.01,
        "search_epochs": 50,
        "formal_epochs": 150,
    },
}

signature_payload = {
    "run_code_version": (
        RUN_CODE_VERSION
    ),
    "models": MODELS,
    "selection_split": (
        FINAL_SELECTION_SPLIT
    ),
    "folds": N_FOLDS,
    "trial_budgets": N_TRIALS,
    "search_spaces": (
        search_space_definition
    ),
    "objective": (
        "arithmetic_mean_of_five_"
        "IL_split_validation_R2_values"
    ),
    "input_manifest_sha256": (
        calculate_sha256(
            INPUT_MANIFEST_PATH
        )
    ),
    "interface_audit_sha256": (
        calculate_sha256(
            INTERFACE_AUDIT_PATH
        )
    ),
    "global_ml_ready_sha256": (
        calculate_sha256(
            ML_READY_PATH
        )
    ),
    "external_10point_read": False,
}

TUNING_SIGNATURE = hashlib.sha256(
    json.dumps(
        to_jsonable(
            signature_payload
        ),
        sort_keys=True,
        ensure_ascii=False,
    ).encode(
        "utf-8"
    )
).hexdigest()

STUDY_SUFFIX = (
    TUNING_SIGNATURE[
        :12
    ]
)

study_plan = pd.DataFrame(
    [
        {
            "Study_number": index,
            "Model": model_name,
            "Split": (
                FINAL_SELECTION_SPLIT
            ),
            "Validation_folds": (
                N_FOLDS
            ),
            "Trial_budget": (
                N_TRIALS[
                    model_name
                ]
            ),
            "Inner_model_fits": (
                N_TRIALS[
                    model_name
                ]
                * N_FOLDS
            ),
            "Objective": "Mean_R2",
            "Study_name": (
                f"Final_{model_name}_"
                f"IL5Fold_{STUDY_SUFFIX}"
            ),
        }
        for (
            index,
            model_name,
        ) in enumerate(
            MODELS,
            start=1,
        )
    ]
)

save_csv_atomic(
    study_plan,
    STUDY_PLAN_PATH,
)

save_json_atomic(
    {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),
        "status": (
            "final_two_model_tuning_"
            "plan_locked"
        ),
        "tuning_signature": (
            TUNING_SIGNATURE
        ),
        "study_suffix": (
            STUDY_SUFFIX
        ),
        "signature_payload": (
            signature_payload
        ),
        "study_plan": (
            study_plan.to_dict(
                orient="records"
            )
        ),
        "optuna_database": str(
            OPTUNA_DB_PATH
        ),
    },
    TUNING_SIGNATURE_PATH,
)

print("=" * 108)
print(
    "Cell 3 最终两模型调参接口与"
    "Study计划锁定完成 ✓"
)
print("=" * 108)

print(
    "tuning signature                 =",
    TUNING_SIGNATURE,
)

print(
    "study suffix                     =",
    STUDY_SUFFIX,
)

print(
    "study count                      =",
    len(study_plan),
)

print(
    "total trials                     =",
    int(
        study_plan[
            "Trial_budget"
        ].sum()
    ),
)

print(
    "total five-fold model fits       =",
    int(
        study_plan[
            "Inner_model_fits"
        ].sum()
    ),
)

print(
    "validation target policy         =",
    "read after prediction frozen only",
)

print(
    "external 10-point file read      =",
    False,
)

print(
    "study plan                       =",
    STUDY_PLAN_PATH,
)

print(
    "tuning signature manifest        =",
    TUNING_SIGNATURE_PATH,
)

display(
    study_plan
)

Cell 3 最终两模型调参接口与Study计划锁定完成 ✓
tuning signature                 = 4659901e5ee2339803adb1263afe0a85ce53fed51b0e2539c7c18cf5adbee163
study suffix                     = 4659901e5ee2
study count                      = 2
total trials                     = 30
total five-fold model fits       = 150
validation target policy         = read after prediction frozen only
external 10-point file read      = False
study plan                       = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\results\final_two_model_study_plan.csv
tuning signature manifest        = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\logs\final_two_model_tuning_signature.json


,Study_number,Model,Split,Validation_folds,Trial_budget,Inner_model_fits,Objective,Study_name
0,1,LightGBM,IL_split,5,20,100,Mean_R2,Final_LightGBM_IL5Fold_4659901e5ee2
1,2,Transformer,IL_split,5,10,50,Mean_R2,Final_Transformer_IL5Fold_4659901e5ee2


In [5]:
# Cell 4 | 运行两模型最终IL_split五折Optuna调参并保存唯一最终参数

def completed_trial_count(
    study,
):
    return sum(
        trial.state
        == optuna.trial.TrialState.COMPLETE
        for trial in study.trials
    )


def make_progress_callback(
    model_name,
    target_trials,
):
    def callback(
        study,
        frozen_trial,
    ):
        complete = (
            completed_trial_count(
                study
            )
        )

        objective = (
            frozen_trial.value
            if frozen_trial.value
            is not None
            else np.nan
        )

        print(
            f"[{model_name}] "
            f"completed {complete}/{target_trials} | "
            f"trial={frozen_trial.number} | "
            f"Mean_R2={objective:.6f}"
        )

    return callback


def create_or_load_study(
    model_name,
):
    study_name = (
        f"Final_{model_name}_"
        f"IL5Fold_{STUDY_SUFFIX}"
    )

    sampler_seed = (
        TUNING_RANDOM_SEED
        + (
            20000
            if model_name
            == "LightGBM"
            else 30000
        )
    )

    storage_url = (
        f"sqlite:///"
        f"{OPTUNA_DB_PATH.as_posix()}"
    )

    return optuna.create_study(
        study_name=study_name,
        storage=storage_url,
        direction="maximize",
        sampler=(
            optuna.samplers.TPESampler(
                seed=sampler_seed,
                multivariate=True,
            )
        ),
        pruner=(
            optuna.pruners.NopPruner()
        ),
        load_if_exists=True,
    )


FINAL_PARAMETER_LOOKUP = {}
tuning_summary_records = []

tuning_start_time = time.time()

for model_name in MODELS:

    study = create_or_load_study(
        model_name
    )

    target_trials = N_TRIALS[
        model_name
    ]

    previous_complete = (
        completed_trial_count(
            study
        )
    )

    while (
        completed_trial_count(
            study
        )
        < target_trials
    ):
        remaining = (
            target_trials
            - completed_trial_count(
                study
            )
        )

        before = (
            completed_trial_count(
                study
            )
        )

        study.optimize(
            lambda trial, current_model=model_name: (
                evaluate_final_trial(
                    trial,
                    current_model,
                )
            ),
            n_trials=remaining,
            callbacks=[
                make_progress_callback(
                    model_name,
                    target_trials,
                )
            ],
            gc_after_trial=True,
            catch=(
                RuntimeError,
                ValueError,
            ),
        )

        after = (
            completed_trial_count(
                study
            )
        )

        if after <= before:
            raise RuntimeError(
                f"{model_name}没有新增完成Trial，"
                "已停止。"
            )

    trial_table = (
        study.trials_dataframe(
            attrs=(
                "number",
                "value",
                "params",
                "user_attrs",
                "state",
            )
        )
    )

    trial_table_path = (
        TRIAL_DIR
        / (
            "final_tuning_trial_table_"
            f"{model_name}.csv"
        )
    )

    save_csv_atomic(
        trial_table,
        trial_table_path,
    )

    best_trial = (
        study.best_trial
    )

    best_search_parameters = dict(
        best_trial.params
    )

    if model_name == "Transformer":
        best_search_parameters[
            "num_tokens"
        ] = (
            TRANSFORMER_NUM_TOKENS
        )

        best_search_parameters[
            "noise_std"
        ] = (
            TRANSFORMER_NOISE_STD
        )

        final_parameters = (
            transformer_search_to_formal(
                best_search_parameters,
                FINAL_MODEL_SEEDS[
                    model_name
                ],
            )
        )

    else:
        final_parameters = (
            lightgbm_search_to_formal(
                best_search_parameters,
                FINAL_MODEL_SEEDS[
                    model_name
                ],
            )
        )

    parameter_payload = {
        "created_at": (
            datetime.now().isoformat(
                timespec="seconds"
            )
        ),
        "run_code_version": (
            RUN_CODE_VERSION
        ),
        "status": (
            "final_full_development_"
            "parameter_ready"
        ),
        "Model": model_name,
        "Model_role": (
            "final_application_model"
            if model_name
            == FINAL_APPLICATION_MODEL
            else (
                "supplementary_external10_"
                "control_only"
            )
        ),
        "parameter_selection_design": (
            "one_new_parameter_set_selected_by_"
            "five_IL_split_development_folds"
        ),
        "outer_fold_parameter_averaging_used": False,
        "single_outer_fold_parameter_reused": False,
        "external_10point_read": False,
        "tuning_signature": (
            TUNING_SIGNATURE
        ),
        "study_name": (
            study.study_name
        ),
        "trial_budget": (
            target_trials
        ),
        "completed_trials": (
            completed_trial_count(
                study
            )
        ),
        "best_trial_number": (
            best_trial.number
        ),
        "best_search_parameters": (
            best_search_parameters
        ),
        "final_training_parameters": (
            final_parameters
        ),
        "best_five_fold_metrics": {
            key: (
                best_trial.user_attrs.get(
                    key
                )
            )
            for key in [
                "Mean_R2",
                "Std_R2",
                "Mean_RMSE",
                "Std_RMSE",
                "Mean_MAE",
                "Std_MAE",
                "Mean_Pearson_r",
                "Std_Pearson_r",
            ]
        },
        "trial_table": str(
            trial_table_path
        ),
        "trial_table_sha256": (
            calculate_sha256(
                trial_table_path
            )
        ),
    }

    parameter_path = (
        PARAMETER_DIR
        / (
            f"final_parameters_"
            f"{model_name}.json"
        )
    )

    save_json_atomic(
        parameter_payload,
        parameter_path,
    )

    FINAL_PARAMETER_LOOKUP[
        model_name
    ] = parameter_payload

    tuning_summary_records.append(
        {
            "Model": model_name,
            "Model_role": (
                parameter_payload[
                    "Model_role"
                ]
            ),
            "Trial_budget": (
                target_trials
            ),
            "Completed_trials_before_this_run": (
                previous_complete
            ),
            "Completed_trials_total": (
                completed_trial_count(
                    study
                )
            ),
            "Best_trial_number": (
                best_trial.number
            ),
            "Mean_R2": (
                best_trial.user_attrs[
                    "Mean_R2"
                ]
            ),
            "Std_R2": (
                best_trial.user_attrs[
                    "Std_R2"
                ]
            ),
            "Mean_RMSE": (
                best_trial.user_attrs[
                    "Mean_RMSE"
                ]
            ),
            "Std_RMSE": (
                best_trial.user_attrs[
                    "Std_RMSE"
                ]
            ),
            "Mean_MAE": (
                best_trial.user_attrs[
                    "Mean_MAE"
                ]
            ),
            "Std_MAE": (
                best_trial.user_attrs[
                    "Std_MAE"
                ]
            ),
            "Mean_Pearson_r": (
                best_trial.user_attrs[
                    "Mean_Pearson_r"
                ]
            ),
            "Std_Pearson_r": (
                best_trial.user_attrs[
                    "Std_Pearson_r"
                ]
            ),
            "Parameter_path": str(
                parameter_path
            ),
            "Trial_table_path": str(
                trial_table_path
            ),
        }
    )

final_tuning_summary = pd.DataFrame(
    tuning_summary_records
)

save_csv_atomic(
    final_tuning_summary,
    TUNING_SUMMARY_PATH,
)

final_tuning_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "run_code_version": (
        RUN_CODE_VERSION
    ),
    "status": (
        "final_two_model_parameter_"
        "selection_completed"
    ),
    "tuning_signature": (
        TUNING_SIGNATURE
    ),
    "selection_design": {
        "models": MODELS,
        "split": (
            FINAL_SELECTION_SPLIT
        ),
        "folds": N_FOLDS,
        "trial_budgets": (
            N_TRIALS
        ),
        "objective": (
            "five_fold_mean_R2"
        ),
        "same_parameter_set_evaluated_"
        "across_all_five_folds_per_trial": True,
        "outer_fold_parameter_"
        "averaging_used": False,
        "single_outer_fold_parameter_"
        "reused": False,
    },
    "leakage_control": {
        "fold_scaler_fitted_only_on_"
        "fold_training_features": True,
        "validation_target_read_after_"
        "prediction_frozen": True,
        "validation_target_used_for_"
        "training": False,
        "validation_target_used_for_"
        "backpropagation": False,
        "validation_target_used_for_"
        "scheduler": False,
        "external_10point_read": False,
    },
    "results": (
        final_tuning_summary.to_dict(
            orient="records"
        )
    ),
    "outputs": {
        "summary": str(
            TUNING_SUMMARY_PATH
        ),
        "parameter_directory": str(
            PARAMETER_DIR
        ),
        "trial_directory": str(
            TRIAL_DIR
        ),
        "optuna_database": str(
            OPTUNA_DB_PATH
        ),
    },
    "elapsed_seconds": (
        time.time()
        - tuning_start_time
    ),
}

save_json_atomic(
    final_tuning_manifest,
    TUNING_MANIFEST_PATH,
)

print("=" * 108)
print(
    "Cell 4 两模型最终IL_split五折调参与"
    "唯一参数保存完成 ✓"
)
print("=" * 108)

print(
    "studies completed                =",
    len(
        final_tuning_summary
    ),
)

print(
    "total completed trials           =",
    int(
        final_tuning_summary[
            "Completed_trials_total"
        ].sum()
    ),
)

print(
    "LightGBM parameter path          =",
    (
        PARAMETER_DIR
        / "final_parameters_LightGBM.json"
    ),
)

print(
    "Transformer parameter path       =",
    (
        PARAMETER_DIR
        / "final_parameters_Transformer.json"
    ),
)

print(
    "tuning summary                   =",
    TUNING_SUMMARY_PATH,
)

print(
    "tuning manifest                  =",
    TUNING_MANIFEST_PATH,
)

print(
    "external 10-point file read      =",
    False,
)

display(
    final_tuning_summary
)

[LightGBM] completed 1/20 | trial=0 | Mean_R2=0.758099
[LightGBM] completed 2/20 | trial=1 | Mean_R2=0.783098
[LightGBM] completed 3/20 | trial=2 | Mean_R2=0.810181
[LightGBM] completed 4/20 | trial=3 | Mean_R2=0.784493
[LightGBM] completed 5/20 | trial=4 | Mean_R2=0.798483
[LightGBM] completed 6/20 | trial=5 | Mean_R2=0.824240
[LightGBM] completed 7/20 | trial=6 | Mean_R2=0.805146
[LightGBM] completed 8/20 | trial=7 | Mean_R2=0.788693
[LightGBM] completed 9/20 | trial=8 | Mean_R2=0.800320
[LightGBM] completed 10/20 | trial=9 | Mean_R2=0.800979
[LightGBM] completed 11/20 | trial=10 | Mean_R2=0.829087
[LightGBM] completed 12/20 | trial=11 | Mean_R2=0.829996
[LightGBM] completed 13/20 | trial=12 | Mean_R2=0.801784
[LightGBM] completed 14/20 | trial=13 | Mean_R2=0.822477
[LightGBM] completed 15/20 | trial=14 | Mean_R2=0.801804
[LightGBM] completed 16/20 | trial=15 | Mean_R2=0.830696
[LightGBM] completed 17/20 | trial=16 | Mean_R2=0.803631
[LightGBM] completed 18/20 | trial=17 | Mean_R2=0.

,Model,Model_role,Trial_budget,Completed_trials_before_this_run,Completed_trials_total,Best_trial_number,Mean_R2,Std_R2,Mean_RMSE,Std_RMSE,Mean_MAE,Std_MAE,Mean_Pearson_r,Std_Pearson_r,Parameter_path,Trial_table_path
0,LightGBM,final_application_model,20,0,20,17,0.832432,0.041205,0.331301,0.038623,0.200850,0.009948,0.914087,0.023083,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...
1,Transformer,supplementary_external10_control_only,10,0,10,3,0.824539,0.032152,0.341493,0.042887,0.194769,0.010686,0.910367,0.017684,D:\jupyter\IL_ML_Project\small_paper_data\visc...,D:\jupyter\IL_ML_Project\small_paper_data\visc...


In [6]:
# Cell 5 | 审计全开发集最终训练输入并拟合唯一特征标准化器

for model_name in MODELS:

    parameter_path = (
        PARAMETER_DIR
        / (
            f"final_parameters_"
            f"{model_name}.json"
        )
    )

    if not parameter_path.exists():
        raise FileNotFoundError(
            "缺少最终参数文件："
            f"{parameter_path}"
        )

    parameter_payload = (
        read_json(
            parameter_path
        )
    )

    if (
        parameter_payload.get(
            "status"
        )
        != (
            "final_full_development_"
            "parameter_ready"
        )
    ):
        raise RuntimeError(
            f"{model_name}最终参数状态不正确。"
        )

    if (
        parameter_payload.get(
            "tuning_signature"
        )
        != TUNING_SIGNATURE
    ):
        raise RuntimeError(
            f"{model_name}最终参数的"
            "调参签名不一致。"
        )

    if bool(
        parameter_payload.get(
            "external_10point_read",
            True,
        )
    ):
        raise RuntimeError(
            f"{model_name}最终参数阶段"
            "读取了外部10点。"
        )

    FINAL_PARAMETER_LOOKUP[
        model_name
    ] = parameter_payload

X_FULL_RAW = np.load(
    GLOBAL_FEATURE_PATH,
    allow_pickle=False,
).astype(
    np.float32
)

target_df = read_existing_columns(
    ML_READY_PATH,
    desired_columns=[
        "eta",
        "lg_eta",
    ],
    required_columns=[
        "lg_eta",
    ],
)

Y_FULL = pd.to_numeric(
    target_df[
        "lg_eta"
    ],
    errors="raise",
).to_numpy(
    dtype=np.float32
)

if (
    X_FULL_RAW.shape
    != (
        EXPECTED_ROWS,
        EXPECTED_FEATURE_DIM,
    )
):
    raise RuntimeError(
        "全开发集特征形状错误："
        f"{X_FULL_RAW.shape}"
    )

if (
    Y_FULL.shape
    != (
        EXPECTED_ROWS,
    )
):
    raise RuntimeError(
        "全开发集目标形状错误："
        f"{Y_FULL.shape}"
    )

if (
    not np.isfinite(
        X_FULL_RAW
    ).all()
    or not np.isfinite(
        Y_FULL
    ).all()
):
    raise RuntimeError(
        "全开发集最终训练输入中存在"
        "NaN或无穷值。"
    )

if "eta" in target_df.columns:

    eta = pd.to_numeric(
        target_df[
            "eta"
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )

    max_log_error = float(
        np.max(
            np.abs(
                np.log10(
                    eta
                )
                - Y_FULL.astype(
                    float
                )
            )
        )
    )

    if max_log_error > 1e-6:
        raise RuntimeError(
            "lg_eta与log10(eta)不一致，"
            f"最大误差={max_log_error}"
        )

else:
    max_log_error = np.nan

FINAL_FEATURE_SCALER = (
    StandardScaler()
)

X_FULL_SCALED = (
    FINAL_FEATURE_SCALER
    .fit_transform(
        X_FULL_RAW
    )
    .astype(
        np.float32
    )
)

if (
    get_scaler_fitted_rows(
        FINAL_FEATURE_SCALER
    )
    != EXPECTED_ROWS
):
    raise RuntimeError(
        "最终特征StandardScaler"
        "拟合行数不是16216。"
    )

if not np.isfinite(
    X_FULL_SCALED
).all():
    raise RuntimeError(
        "标准化后的全开发集特征中"
        "存在NaN或无穷值。"
    )

save_joblib_atomic(
    FINAL_FEATURE_SCALER,
    FINAL_SCALER_PATH,
)

full_input_audit_df = pd.DataFrame(
    [
        {
            "Input_role": (
                "development_features"
            ),
            "Path": str(
                GLOBAL_FEATURE_PATH
            ),
            "Rows": (
                X_FULL_RAW.shape[0]
            ),
            "Columns": (
                X_FULL_RAW.shape[1]
            ),
            "SHA256": (
                calculate_sha256(
                    GLOBAL_FEATURE_PATH
                )
            ),
            "Used_for_final_training": True,
            "External_10point": False,
        },
        {
            "Input_role": (
                "development_targets"
            ),
            "Path": str(
                ML_READY_PATH
            ),
            "Rows": len(
                Y_FULL
            ),
            "Columns": 1,
            "SHA256": (
                calculate_sha256(
                    ML_READY_PATH
                )
            ),
            "Used_for_final_training": True,
            "External_10point": False,
        },
        {
            "Input_role": (
                "final_feature_scaler"
            ),
            "Path": str(
                FINAL_SCALER_PATH
            ),
            "Rows": (
                get_scaler_fitted_rows(
                    FINAL_FEATURE_SCALER
                )
            ),
            "Columns": (
                EXPECTED_FEATURE_DIM
            ),
            "SHA256": (
                calculate_sha256(
                    FINAL_SCALER_PATH
                )
            ),
            "Used_for_final_training": True,
            "External_10point": False,
        },
    ]
)

save_csv_atomic(
    full_input_audit_df,
    FULL_INPUT_AUDIT_CSV_PATH,
)

full_input_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "run_code_version": (
        RUN_CODE_VERSION
    ),
    "status": (
        "full_development_training_"
        "inputs_verified"
    ),
    "development_rows": (
        EXPECTED_ROWS
    ),
    "feature_dim": (
        EXPECTED_FEATURE_DIM
    ),
    "target": "lg_eta",
    "feature_source": (
        "frozen_global_MPNN_representation_"
        "for_16216_development_rows"
    ),
    "mpnn_extractor_retrained_in_"
    "this_notebook": False,
    "mpnn_extractor_external_10point_"
    "used_in_pretraining": False,
    "downstream_scaler_fitted_rows": (
        get_scaler_fitted_rows(
            FINAL_FEATURE_SCALER
        )
    ),
    "lg_eta_log10_eta_max_error": (
        max_log_error
    ),
    "external_10point_file_read": False,
    "inputs": (
        full_input_audit_df.to_dict(
            orient="records"
        )
    ),
    "outputs": {
        "input_audit_csv": str(
            FULL_INPUT_AUDIT_CSV_PATH
        ),
        "input_audit_json": str(
            FULL_INPUT_AUDIT_JSON_PATH
        ),
        "feature_scaler": str(
            FINAL_SCALER_PATH
        ),
    },
}

save_json_atomic(
    full_input_manifest,
    FULL_INPUT_AUDIT_JSON_PATH,
)

print("=" * 108)
print(
    "Cell 5 全开发集最终训练输入与"
    "唯一特征标准化器审计通过 ✓"
)
print("=" * 108)

print(
    "X_FULL_RAW shape                  =",
    X_FULL_RAW.shape,
)

print(
    "X_FULL_SCALED shape               =",
    X_FULL_SCALED.shape,
)

print(
    "Y_FULL shape                      =",
    Y_FULL.shape,
)

print(
    "scaler fitted rows                =",
    get_scaler_fitted_rows(
        FINAL_FEATURE_SCALER
    ),
)

print(
    "MPNN extractor retrained here     =",
    False,
)

print(
    "external 10-point file read       =",
    False,
)

print(
    "final feature scaler              =",
    FINAL_SCALER_PATH,
)

print(
    "full input audit CSV              =",
    FULL_INPUT_AUDIT_CSV_PATH,
)

print(
    "full input audit JSON             =",
    FULL_INPUT_AUDIT_JSON_PATH,
)

display(
    full_input_audit_df
)

Cell 5 全开发集最终训练输入与唯一特征标准化器审计通过 ✓
X_FULL_RAW shape                  = (16216, 130)
X_FULL_SCALED shape               = (16216, 130)
Y_FULL shape                      = (16216,)
scaler fitted rows                = 16216
MPNN extractor retrained here     = False
external 10-point file read       = False
final feature scaler              = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\scalers\final_downstream_feature_scaler.joblib
full input audit CSV              = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\logs\full_development_training_input_audit.csv
full input audit JSON             = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\logs\full_development_training_input_audit.json


,Input_role,Path,Rows,Columns,SHA256,Used_for_final_training,External_10point
0,development_features,D:\jupyter\IL_ML_Project\small_paper_data\visc...,16216,130,d4f474817c32c41e9ff20ff6377bff7a4aa45463ca2183...,True,False
1,development_targets,D:\jupyter\IL_ML_Project\small_paper_data\visc...,16216,1,bedae296be0e14d654747a7e57ae6364b17b723cc3dfd0...,True,False
2,final_feature_scaler,D:\jupyter\IL_ML_Project\small_paper_data\visc...,16216,130,b51e21e80c03f71f3749a0aad3d1c0b2cf11b2e87fd332...,True,False


In [7]:
# Cell 6 | 使用全部16216条开发数据训练并保存最终LightGBM和Transformer

training_summary_records = []

lgbm_payload = (
    FINAL_PARAMETER_LOOKUP[
        "LightGBM"
    ]
)

lgbm_parameters = (
    lgbm_payload[
        "final_training_parameters"
    ]
)

lgbm_start = time.time()

FINAL_LGBM_MODEL = (
    LGBMRegressor(
        **lgbm_parameters
    )
)

FINAL_LGBM_MODEL.fit(
    X_FULL_SCALED,
    Y_FULL,
)

lgbm_train_prediction = (
    FINAL_LGBM_MODEL.predict(
        X_FULL_SCALED
    )
)

lgbm_train_metrics = (
    calculate_metrics(
        Y_FULL,
        lgbm_train_prediction,
    )
)

save_joblib_atomic(
    FINAL_LGBM_MODEL,
    FINAL_LGBM_PATH,
)

training_summary_records.append(
    {
        "Model": "LightGBM",
        "Model_role": (
            "final_application_model_for_"
            "external10_and_virtual_IL_prediction"
        ),
        "Training_rows": (
            EXPECTED_ROWS
        ),
        "Feature_dim": (
            EXPECTED_FEATURE_DIM
        ),
        "Formal_epochs": np.nan,
        "Train_R2_diagnostic_only": (
            lgbm_train_metrics[
                "R2"
            ]
        ),
        "Train_RMSE_diagnostic_only": (
            lgbm_train_metrics[
                "RMSE"
            ]
        ),
        "Train_MAE_diagnostic_only": (
            lgbm_train_metrics[
                "MAE"
            ]
        ),
        "Train_Pearson_r_diagnostic_only": (
            lgbm_train_metrics[
                "Pearson_r"
            ]
        ),
        "Training_seconds": (
            time.time()
            - lgbm_start
        ),
        "Model_path": str(
            FINAL_LGBM_PATH
        ),
        "Model_sha256": (
            calculate_sha256(
                FINAL_LGBM_PATH
            )
        ),
        "Use_for_virtual_IL_prediction": True,
    }
)

transformer_payload = (
    FINAL_PARAMETER_LOOKUP[
        "Transformer"
    ]
)

transformer_parameters = (
    transformer_payload[
        "final_training_parameters"
    ]
)

transformer_start = time.time()

(
    FINAL_TRANSFORMER_MODEL,
    transformer_history,
) = fit_transformer_core(
    X_train=(
        X_FULL_SCALED
    ),
    y_train=(
        Y_FULL
    ),
    parameters=(
        transformer_parameters
    ),
    device=DEVICE,
    epochs=int(
        transformer_parameters[
            "formal_epochs"
        ]
    ),
    seed=int(
        transformer_parameters[
            "seed"
        ]
    ),
)

transformer_train_prediction = (
    predict_transformer_in_batches(
        FINAL_TRANSFORMER_MODEL,
        X_FULL_SCALED,
        DEVICE,
    )
)

transformer_train_metrics = (
    calculate_metrics(
        Y_FULL,
        transformer_train_prediction,
    )
)

transformer_history_df = pd.DataFrame(
    transformer_history
)

save_csv_atomic(
    transformer_history_df,
    FINAL_TRANSFORMER_HISTORY_PATH,
)

transformer_checkpoint = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "run_code_version": (
        RUN_CODE_VERSION
    ),
    "status": (
        "final_transformer_frozen"
    ),
    "model_state_dict": {
        key: value.detach().cpu()
        for (
            key,
            value,
        ) in (
            FINAL_TRANSFORMER_MODEL
            .state_dict()
            .items()
        )
    },
    "formal_parameters": (
        transformer_parameters
    ),
    "training_rows": (
        EXPECTED_ROWS
    ),
    "feature_dim": (
        EXPECTED_FEATURE_DIM
    ),
    "target": "lg_eta",
    "feature_scaler_path": str(
        FINAL_SCALER_PATH
    ),
    "feature_scaler_sha256": (
        calculate_sha256(
            FINAL_SCALER_PATH
        )
    ),
    "global_feature_sha256": (
        calculate_sha256(
            GLOBAL_FEATURE_PATH
        )
    ),
    "external_10point_read": False,
}

save_torch_atomic(
    transformer_checkpoint,
    FINAL_TRANSFORMER_PATH,
)

training_summary_records.append(
    {
        "Model": "Transformer",
        "Model_role": (
            "supplementary_external10_"
            "control_only"
        ),
        "Training_rows": (
            EXPECTED_ROWS
        ),
        "Feature_dim": (
            EXPECTED_FEATURE_DIM
        ),
        "Formal_epochs": int(
            transformer_parameters[
                "formal_epochs"
            ]
        ),
        "Train_R2_diagnostic_only": (
            transformer_train_metrics[
                "R2"
            ]
        ),
        "Train_RMSE_diagnostic_only": (
            transformer_train_metrics[
                "RMSE"
            ]
        ),
        "Train_MAE_diagnostic_only": (
            transformer_train_metrics[
                "MAE"
            ]
        ),
        "Train_Pearson_r_diagnostic_only": (
            transformer_train_metrics[
                "Pearson_r"
            ]
        ),
        "Training_seconds": (
            time.time()
            - transformer_start
        ),
        "Model_path": str(
            FINAL_TRANSFORMER_PATH
        ),
        "Model_sha256": (
            calculate_sha256(
                FINAL_TRANSFORMER_PATH
            )
        ),
        "Use_for_virtual_IL_prediction": False,
    }
)

final_training_summary = pd.DataFrame(
    training_summary_records
)

save_csv_atomic(
    final_training_summary,
    FINAL_TRAINING_SUMMARY_PATH,
)

print("=" * 108)
print(
    "Cell 6 两个最终全开发集模型"
    "训练与保存完成 ✓"
)
print("=" * 108)

print(
    "final LightGBM                   =",
    FINAL_LGBM_PATH,
)

print(
    "final Transformer                =",
    FINAL_TRANSFORMER_PATH,
)

print(
    "Transformer history              =",
    FINAL_TRANSFORMER_HISTORY_PATH,
)

print(
    "training summary                 =",
    FINAL_TRAINING_SUMMARY_PATH,
)

print(
    "external 10-point file read      =",
    False,
)

print(
    "注意：训练集指标仅用于文件与"
    "训练过程审计，不作为论文泛化性能。"
)

display(
    final_training_summary
)

Cell 6 两个最终全开发集模型训练与保存完成 ✓
final LightGBM                   = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\models\final_LightGBM_viscosity.joblib
final Transformer                = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\models\final_Transformer_viscosity.pt
Transformer history              = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\results\final_Transformer_training_history.csv
training summary                 = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\results\final_model_training_summary.csv
external 10-point file read      = False
注意：训练集指标仅用于文件与训练过程审计，不作为论文泛化性能。


,Model,Model_role,Training_rows,Feature_dim,Formal_epochs,Train_R2_diagnostic_only,Train_RMSE_diagnostic_only,Train_MAE_diagnostic_only,Train_Pearson_r_diagnostic_only,Training_seconds,Model_path,Model_sha256,Use_for_virtual_IL_prediction
0,LightGBM,final_application_model_for_external10_and_vir...,16216,130,NaN,0.979775,0.118883,0.076802,0.990151,3.473650,D:\jupyter\IL_ML_Project\small_paper_data\visc...,29d4cbcefa0896225f413ac397cb85cf807185a74b4340...,True
1,Transformer,supplementary_external10_control_only,16216,130,150.0,0.992995,0.069967,0.040172,0.996520,189.089307,D:\jupyter\IL_ML_Project\small_paper_data\visc...,6e03e4dd0e5b9342a1a776770e76abee252cea4261f56b...,False


In [8]:
# Cell 7 | 完成两模型冻结、用途登记和全部输出完整性审计

required_outputs = [
    INPUT_AUDIT_PATH,
    INPUT_MANIFEST_PATH,
    INTERFACE_AUDIT_PATH,
    TUNING_SIGNATURE_PATH,
    STUDY_PLAN_PATH,
    TUNING_SUMMARY_PATH,
    TUNING_MANIFEST_PATH,
    (
        PARAMETER_DIR
        / "final_parameters_LightGBM.json"
    ),
    (
        PARAMETER_DIR
        / "final_parameters_Transformer.json"
    ),
    FINAL_SCALER_PATH,
    FULL_INPUT_AUDIT_CSV_PATH,
    FULL_INPUT_AUDIT_JSON_PATH,
    FINAL_LGBM_PATH,
    FINAL_TRANSFORMER_PATH,
    FINAL_TRANSFORMER_HISTORY_PATH,
    FINAL_TRAINING_SUMMARY_PATH,
]

output_check_records = []

for path in required_outputs:

    path = Path(path)

    output_check_records.append(
        {
            "File_name": (
                path.name
            ),
            "Path": str(
                path
            ),
            "Exists": (
                path.exists()
            ),
            "Size_bytes": (
                int(
                    path.stat().st_size
                )
                if path.exists()
                else 0
            ),
            "SHA256": (
                calculate_sha256(
                    path
                )
                if path.exists()
                else None
            ),
        }
    )

output_check_df = pd.DataFrame(
    output_check_records
)

if not output_check_df[
    "Exists"
].all():
    display(
        output_check_df.loc[
            ~output_check_df[
                "Exists"
            ]
        ]
    )

    raise RuntimeError(
        "最终冻结阶段仍有必要输出缺失。"
    )

loaded_scaler = joblib.load(
    FINAL_SCALER_PATH
)

if (
    get_scaler_fitted_rows(
        loaded_scaler
    )
    != EXPECTED_ROWS
):
    raise RuntimeError(
        "重新加载后的最终特征标准化器"
        "拟合行数错误。"
    )

loaded_lgbm = joblib.load(
    FINAL_LGBM_PATH
)

lgbm_dummy = np.asarray(
    loaded_lgbm.predict(
        np.zeros(
            (
                2,
                130,
            ),
            dtype=np.float32,
        )
    )
).reshape(-1)

if (
    lgbm_dummy.shape
    != (
        2,
    )
    or not np.isfinite(
        lgbm_dummy
    ).all()
):
    raise RuntimeError(
        "重新加载后的LightGBM前向测试失败。"
    )

try:
    loaded_checkpoint = torch.load(
        FINAL_TRANSFORMER_PATH,
        map_location="cpu",
        weights_only=False,
    )

except TypeError:
    loaded_checkpoint = torch.load(
        FINAL_TRANSFORMER_PATH,
        map_location="cpu",
    )

loaded_transformer_parameters = (
    loaded_checkpoint[
        "formal_parameters"
    ]
)

loaded_transformer = (
    build_transformer_model(
        loaded_transformer_parameters
    )
)

loaded_transformer.load_state_dict(
    loaded_checkpoint[
        "model_state_dict"
    ]
)

loaded_transformer.eval()

with torch.no_grad():
    transformer_dummy = (
        loaded_transformer(
            torch.zeros(
                (
                    2,
                    130,
                ),
                dtype=torch.float32,
            )
        )
        .numpy()
        .reshape(-1)
    )

if (
    transformer_dummy.shape
    != (
        2,
    )
    or not np.isfinite(
        transformer_dummy
    ).all()
):
    raise RuntimeError(
        "重新加载后的Transformer前向测试失败。"
    )

model_registry = pd.DataFrame(
    [
        {
            "Model": "LightGBM",
            "Model_label": "LGBM",
            "Model_role": (
                "final_application_model"
            ),
            "Model_path": str(
                FINAL_LGBM_PATH
            ),
            "Model_sha256": (
                calculate_sha256(
                    FINAL_LGBM_PATH
                )
            ),
            "Parameter_path": str(
                PARAMETER_DIR
                / "final_parameters_LightGBM.json"
            ),
            "Parameter_sha256": (
                calculate_sha256(
                    PARAMETER_DIR
                    / "final_parameters_LightGBM.json"
                )
            ),
            "Scaler_path": str(
                FINAL_SCALER_PATH
            ),
            "Scaler_sha256": (
                calculate_sha256(
                    FINAL_SCALER_PATH
                )
            ),
            "Use_for_external_10point_"
            "supplementary_validation": True,
            "Use_for_virtual_IL_"
            "viscosity_prediction": True,
            "May_be_reselected_by_"
            "external_10point": False,
        },
        {
            "Model": "Transformer",
            "Model_label": "Transformer",
            "Model_role": (
                "supplementary_control_model"
            ),
            "Model_path": str(
                FINAL_TRANSFORMER_PATH
            ),
            "Model_sha256": (
                calculate_sha256(
                    FINAL_TRANSFORMER_PATH
                )
            ),
            "Parameter_path": str(
                PARAMETER_DIR
                / "final_parameters_Transformer.json"
            ),
            "Parameter_sha256": (
                calculate_sha256(
                    PARAMETER_DIR
                    / "final_parameters_Transformer.json"
                )
            ),
            "Scaler_path": str(
                FINAL_SCALER_PATH
            ),
            "Scaler_sha256": (
                calculate_sha256(
                    FINAL_SCALER_PATH
                )
            ),
            "Use_for_external_10point_"
            "supplementary_validation": True,
            "Use_for_virtual_IL_"
            "viscosity_prediction": False,
            "May_be_reselected_by_"
            "external_10point": False,
        },
    ]
)

save_csv_atomic(
    model_registry,
    FINAL_REGISTRY_PATH,
)

output_check_df = pd.concat(
    [
        output_check_df,
        pd.DataFrame(
            [
                {
                    "File_name": (
                        FINAL_REGISTRY_PATH.name
                    ),
                    "Path": str(
                        FINAL_REGISTRY_PATH
                    ),
                    "Exists": (
                        FINAL_REGISTRY_PATH.exists()
                    ),
                    "Size_bytes": int(
                        FINAL_REGISTRY_PATH
                        .stat()
                        .st_size
                    ),
                    "SHA256": (
                        calculate_sha256(
                            FINAL_REGISTRY_PATH
                        )
                    ),
                }
            ]
        ),
    ],
    ignore_index=True,
)

save_csv_atomic(
    output_check_df,
    OUTPUT_CHECK_PATH,
)

freeze_manifest = {
    "created_at": (
        datetime.now().isoformat(
            timespec="seconds"
        )
    ),
    "run_code_version": (
        RUN_CODE_VERSION
    ),
    "status": (
        "final_two_model_freeze_completed"
    ),
    "formal_selection": {
        "final_application_model": (
            "LightGBM"
        ),
        "formal_selection_source": str(
            FORMAL_FINAL_MANIFEST_PATH
        ),
        "formal_selection_source_sha256": (
            calculate_sha256(
                FORMAL_FINAL_MANIFEST_PATH
            )
        ),
        "selection_changed_in_"
        "this_notebook": False,
        "Data_split_and_IL_split_"
        "averaged": False,
    },
    "final_parameter_selection": {
        "design": (
            "new_single_parameter_set_per_model_"
            "from_five_IL_split_development_folds"
        ),
        "tuning_signature": (
            TUNING_SIGNATURE
        ),
        "outer_fold_parameter_"
        "averaging_used": False,
        "single_outer_fold_parameter_"
        "reused": False,
        "external_10point_read": False,
    },
    "final_training": {
        "development_rows": (
            EXPECTED_ROWS
        ),
        "feature_dim": (
            EXPECTED_FEATURE_DIM
        ),
        "target": "lg_eta",
        "global_MPNN_representation_path": str(
            GLOBAL_FEATURE_PATH
        ),
        "global_MPNN_representation_sha256": (
            calculate_sha256(
                GLOBAL_FEATURE_PATH
            )
        ),
        "MPNN_extractor_retrained_in_"
        "this_notebook": False,
        "MPNN_external_10point_used_in_"
        "pretraining": False,
        "downstream_scaler_fitted_on_all_"
        "development_rows": True,
        "LightGBM_trained_on_all_"
        "development_rows": True,
        "Transformer_trained_on_all_"
        "development_rows": True,
    },
    "model_roles": (
        model_registry.to_dict(
            orient="records"
        )
    ),
    "usage_lock": {
        "external_10point_role": (
            "supplementary_validation_only"
        ),
        "external_10point_may_change_"
        "final_model": False,
        "virtual_IL_prediction_model": (
            "LightGBM"
        ),
        "Transformer_used_for_virtual_"
        "IL_prediction": False,
    },
    "verification": {
        "all_required_outputs_exist": bool(
            output_check_df[
                "Exists"
            ].all()
        ),
        "final_scaler_reload_verified": True,
        "final_LightGBM_reload_verified": True,
        "final_Transformer_reload_verified": True,
        "external_10point_file_read": False,
        "formal_outer_metrics_recomputed_"
        "or_replaced": False,
        "figure_catalog_created": False,
        "table_catalog_created": False,
    },
    "outputs": {
        "model_registry": str(
            FINAL_REGISTRY_PATH
        ),
        "output_check": str(
            OUTPUT_CHECK_PATH
        ),
        "freeze_manifest": str(
            FINAL_FREEZE_MANIFEST_PATH
        ),
    },
}

save_json_atomic(
    freeze_manifest,
    FINAL_FREEZE_MANIFEST_PATH,
)

print("=" * 108)
print(
    "Cell 7 两模型最终冻结和"
    "输出完整性审计完成 ✓"
)
print("=" * 108)

print(
    "all required outputs exist            =",
    bool(
        output_check_df[
            "Exists"
        ].all()
    ),
)

print(
    "final scaler reload verified          =",
    True,
)

print(
    "final LightGBM reload verified        =",
    True,
)

print(
    "final Transformer reload verified     =",
    True,
)

print(
    "final application model               =",
    "LightGBM",
)

print(
    "Transformer role                      =",
    "supplementary control only",
)

print(
    "virtual IL prediction model           =",
    "LightGBM",
)

print(
    "external 10-point may reselect model  =",
    False,
)

print(
    "external 10-point file read           =",
    False,
)

print(
    "model registry                        =",
    FINAL_REGISTRY_PATH,
)

print(
    "output check                          =",
    OUTPUT_CHECK_PATH,
)

print(
    "freeze manifest                       =",
    FINAL_FREEZE_MANIFEST_PATH,
)

print(
    "最终状态："
    "final_two_model_freeze_completed"
)

display(
    model_registry
)

display(
    output_check_df
)

Cell 7 两模型最终冻结和输出完整性审计完成 ✓
all required outputs exist            = True
final scaler reload verified          = True
final LightGBM reload verified        = True
final Transformer reload verified     = True
final application model               = LightGBM
Transformer role                      = supplementary control only
virtual IL prediction model           = LightGBM
external 10-point may reselect model  = False
external 10-point file read           = False
model registry                        = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\results\final_two_model_registry.csv
output check                          = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\model_training_3models_cv_no_leakage_2splits\final_two_model_freeze\logs\final_two_model_output_check.csv
freeze manifest                       = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_1

,Model,Model_label,Model_role,Model_path,Model_sha256,Parameter_path,Parameter_sha256,Scaler_path,Scaler_sha256,Use_for_external_10point_supplementary_validation,Use_for_virtual_IL_viscosity_prediction,May_be_reselected_by_external_10point
0,LightGBM,LGBM,final_application_model,D:\jupyter\IL_ML_Project\small_paper_data\visc...,29d4cbcefa0896225f413ac397cb85cf807185a74b4340...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,2c14323e0366a7fb44c11bcd608fa052c050a9b42c92df...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,b51e21e80c03f71f3749a0aad3d1c0b2cf11b2e87fd332...,True,True,False
1,Transformer,Transformer,supplementary_control_model,D:\jupyter\IL_ML_Project\small_paper_data\visc...,6e03e4dd0e5b9342a1a776770e76abee252cea4261f56b...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,61e0c27549d20a71b72d820ab0e5398c1c5cd1bf3d5315...,D:\jupyter\IL_ML_Project\small_paper_data\visc...,b51e21e80c03f71f3749a0aad3d1c0b2cf11b2e87fd332...,True,False,False


,File_name,Path,Exists,Size_bytes,SHA256
0,final_two_model_input_audit.csv,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,474,d9f69ebb29c12d87efec1c109de4cc52100e39f67c6beb...
1,final_two_model_input_manifest.json,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,5868,27319d4f5c038d4188dfd04e155c487b5271a9517eb1c4...
2,final_two_model_training_interface_audit.json,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,955,bfa5dd83f41342f0abd859bc9772bcd5a6383b83d3b979...
3,final_two_model_tuning_signature.json,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,3337,b4a3533b9e04f2ba232b55bc561d96e9418a3018e69559...
4,final_two_model_study_plan.csv,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,250,304de34d5d40db3ef2911ffb1e8eb35edfca1ac8d49281...
5,final_two_model_tuning_summary.csv,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,1415,9ca6e698f2db14a5b98c2edbf15b7ab6a152ddbdc25d7d...
6,final_two_model_tuning_manifest.json,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,4026,3a5dbf990c155370c7bf6e2a0deec4eb93d40a0f766fda...
7,final_parameters_LightGBM.json,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,2339,2c14323e0366a7fb44c11bcd608fa052c050a9b42c92df...
8,final_parameters_Transformer.json,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,2062,61e0c27549d20a71b72d820ab0e5398c1c5cd1bf3d5315...
9,final_downstream_feature_scaler.joblib,D:\jupyter\IL_ML_Project\small_paper_data\visc...,True,3735,b51e21e80c03f71f3749a0aad3d1c0b2cf11b2e87fd332...
